# SmokeGuard — Weighted Late Fusion (deployed backend contract)

This notebook combines two already-trained models into one decision. It now
mirrors the **deployed backend** (`app/config.py`, `app/inference.py`,
`app/detection.py`), not a separate research policy.

| Modality | Notebook | Output used here |
|---|---|---|
| CSI gesture model (1D ResNet-LSTM) | `SmokeGuard_v2.ipynb` | calibrated seven-class probabilities; `P(smoke)` and calibrated argmax |
| Smoke sensor (RBF SVM) | `SmokeGuard_Smoke_Sensor.ipynb` | calibrated binary `P(smoke)`; positive class `smoke` |

## Deployed decision policy

```text
csi_hit    = argmax(P_csi_calibrated) == 'smoke'
sensor_hit = P_sensor_smoke >= SENSOR_THRESHOLD     # smoke_svm.meta.json (currently 0.404999)
score      = 0.50 * P_csi_smoke + 0.50 * P_sensor_smoke

fusion     = csi_hit AND sensor_hit AND score >= 0.70
smoke_only = sensor_hit AND NOT fusion              # sensor-only "possible" alert
csi_only   = experimental, OFF by default: csi_only_enabled AND
             p_sensor is not None AND p_sensor < sensor gate AND
             csi_hit AND p_csi >= csi_only_min_prob
```

`fused_score` is reported for `fusion` and `smoke_only`; it is `null` for
`csi_only` (no fusion was computed). All three labels produce the backend's
WebSocket `alert`; `fusion` is the confirmed alarm, while `smoke_only` and
`csi_only` are "possible" alerts with distinct UI styling.

| Situation | Deployed outcome |
|---|---|
| CSI gesture + sensor smoke + score >= 0.70 | **fusion** (confirmed alert) |
| No CSI gesture + sensor smoke | **smoke_only** (possible alert) |
| CSI gesture + sensor smoke but score < 0.70 | **smoke_only** |
| CSI gesture + sensor silent | no alert (or **csi_only** if experimentally enabled) |
| Actually smoking + sensor missed | no alert (accepted compromise) |
| CSI false positive (e.g. drinking) + sensor silent | no alert (or **csi_only** if enabled) |

## Two kinds of evidence in this notebook

1. **Synthetic rule tests** prove the policy logic structurally; they do not use model data.
2. **Scenario simulations** use the actual held-out test sets of both models. Because
   the CSI and sensor recordings are not synchronized, these pair CSI test windows with
   sensor test intervals by label/outcome. They are **simulations**, not a joint evaluation.
   Every scenario table is marked `simulated=True`, and no fused accuracy/ROC is claimed.

**Critical limitation:** there are no synchronized CSI + sensor recordings yet. Fused
accuracy, precision/recall/F1, and optimal weights/thresholds cannot be measured until
paired data exist. A ready-to-use evaluation harness is included at the end.


## 1. Environment setup

Mount Drive and define every path used by the notebook. All model artifacts and test
fixtures live under `MyDrive/CSI_TRAINING/`.

In [ ]:
# @title 1. Environment setup
import hashlib
import json
import os
import shutil
import tempfile
import warnings
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

try:
    from google.colab import drive  # type: ignore
    if os.path.isdir('/content/drive/MyDrive'):
        print('[setup] Google Drive already mounted.')
    else:
        drive.mount('/content/drive')
except Exception as exc:
    print(f'[setup] Drive mount skipped or failed: {exc}')

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.max_colwidth', 120)

BASE_DIR = '/content/drive/MyDrive/CSI_TRAINING'
MODEL_DIR = os.path.join(BASE_DIR, 'models')
CSI_MODEL_PATH = os.path.join(MODEL_DIR, 'csi_resnet_lstm_7class.keras')
CSI_META_PATH = os.path.join(MODEL_DIR, 'csi_resnet_lstm_7class.meta.json')
CSI_PLATT_PATH = os.path.join(MODEL_DIR, 'csi_resnet_lstm_7class.platt.json')
SENSOR_MODEL_DIR = os.path.join(MODEL_DIR, 'smoke_sensor_svm_v2')
SENSOR_PIPELINE_PATH = os.path.join(SENSOR_MODEL_DIR, 'smoke_svm_pipeline.joblib')
SENSOR_META_PATH = os.path.join(SENSOR_MODEL_DIR, 'smoke_svm.meta.json')

FIXTURE_DIR = os.path.join(BASE_DIR, 'fusion', 'fixtures')
CSI_FIXTURE_PATH = os.path.join(FIXTURE_DIR, 'csi_test_fixture.npz')
CSI_FIXTURE_META_PATH = os.path.join(FIXTURE_DIR, 'csi_test_fixture.meta.json')
SENSOR_FIXTURE_PATH = os.path.join(FIXTURE_DIR, 'sensor_test_fixture.npz')
SENSOR_FIXTURE_META_PATH = os.path.join(FIXTURE_DIR, 'sensor_test_fixture.meta.json')

FUSION_DIR = os.path.join(MODEL_DIR, 'fusion_v1')
FUSION_HISTORY_DIR = os.path.join(FUSION_DIR, 'history')
FUSION_POLICY_PATH = os.path.join(FUSION_DIR, 'fusion_policy.json')
FUSION_META_PATH = os.path.join(FUSION_DIR, 'fusion_meta.json')
FUSION_REPORT_PATH = os.path.join(FUSION_DIR, 'fusion_report.json')
FUSION_SCENARIOS_CSV_PATH = os.path.join(FUSION_DIR, 'fusion_scenarios.csv')
FUSION_SCENARIOS_JSON_PATH = os.path.join(FUSION_DIR, 'fusion_scenarios.json')

SEVEN_CLASSES = ['smoke', 'eat', 'drink', 'sit', 'stand', 'random', 'static']
SENSOR_CLASSES = ['no-smoke', 'smoke']
FUSION_SCHEMA_VERSION = 2
EXPECTED_CSI_INPUT_SHAPE = (420, 52)

# Deployed policy (backend ground truth: app/config.py + app/inference.py)
FUSION_CSI_WEIGHT = 0.50
FUSION_SENSOR_WEIGHT = 1.0 - FUSION_CSI_WEIGHT
FUSION_SCORE_THRESHOLD = 0.70
CSI_ONLY_ENABLED = False          # experimental; off by default in deployment
CSI_ONLY_MIN_PROB = 0.0


def utc_now_iso() -> str:
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')


def sha256_file(path: str, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(chunk_size), b''):
            digest.update(block)
    return digest.hexdigest()


def atomic_write_text(path: str, text: str) -> str:
    """Drive-safe write: local temp file, copy, then read-back verification."""
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    handle_fd, local_path = tempfile.mkstemp(prefix='fusion_', suffix='.tmp')
    try:
        with os.fdopen(handle_fd, 'w', encoding='utf-8') as handle:
            handle.write(text)
        shutil.copy2(local_path, path)
    finally:
        if os.path.exists(local_path):
            os.remove(local_path)
    with open(path, 'r', encoding='utf-8') as handle:
        saved = handle.read()
    if saved != text:
        raise RuntimeError(f'Drive write verification failed for {path}.')
    return hashlib.sha256(saved.encode('utf-8')).hexdigest()


def atomic_write_json(path: str, payload: dict) -> str:
    text = json.dumps(payload, indent=2, sort_keys=True) + '\n'
    return atomic_write_text(path, text)


def frame_records(frame: pd.DataFrame) -> list:
    """JSON-safe records (converts numpy scalars and NaN to null)."""
    return json.loads(frame.to_json(orient='records'))


print(f'[setup] TensorFlow {tf.__version__} | Keras '
      f'{getattr(keras, "__version__", "unknown")}')
print(f'[setup] NumPy {np.__version__} | pandas {pd.__version__}')
print(f'[setup] Base directory: {BASE_DIR}')
print(f'[setup] Fusion policy: {FUSION_CSI_WEIGHT:.2f} CSI / '
      f'{FUSION_SENSOR_WEIGHT:.2f} sensor, fused threshold '
      f'{FUSION_SCORE_THRESHOLD:.2f}, csi_only={CSI_ONLY_ENABLED}')


[setup] Drive mount skipped or failed: mount failed
[setup] TensorFlow 2.21.0 | Keras 3.13.2
[setup] NumPy 2.4.6 | pandas 2.2.3
[setup] Base directory: /content/drive/MyDrive/CSI_TRAINING
[setup] Fusion policy: 0.50 CSI / 0.50 sensor, fused threshold 0.70, csi_only=False


## 2. Artifact preflight

This cell is the first thing to run in Colab. It verifies the two saved models and the
two held-out test fixtures. Missing model artifacts stop execution; missing fixtures
print the exact export instructions and the simulation section is skipped.

In [ ]:
# @title 2. Artifact preflight
ARTIFACT_PATHS = {
    'CSI model': CSI_MODEL_PATH,
    'CSI metadata': CSI_META_PATH,
    'CSI Platt': CSI_PLATT_PATH,
    'Sensor pipeline': SENSOR_PIPELINE_PATH,
    'Sensor metadata': SENSOR_META_PATH,
    'CSI test fixture': CSI_FIXTURE_PATH,
    'CSI fixture metadata': CSI_FIXTURE_META_PATH,
    'Sensor test fixture': SENSOR_FIXTURE_PATH,
    'Sensor fixture metadata': SENSOR_FIXTURE_META_PATH,
}

preflight_rows = []
for label, path in ARTIFACT_PATHS.items():
    row = {'artifact': label, 'path': path, 'exists': os.path.exists(path)}
    if row['exists']:
        row['size_bytes'] = os.path.getsize(path)
        row['modified_utc'] = datetime.fromtimestamp(
            os.path.getmtime(path), tz=timezone.utc
        ).strftime('%Y-%m-%dT%H:%M:%SZ')
        row['sha256'] = sha256_file(path)
    preflight_rows.append(row)
PREFLIGHT = pd.DataFrame(preflight_rows)
display(PREFLIGHT)

CSI_PLATT_USED_PATH = CSI_PLATT_PATH
REQUIRED_MODEL_ARTIFACTS = [
    CSI_MODEL_PATH, CSI_META_PATH, CSI_PLATT_USED_PATH,
    SENSOR_PIPELINE_PATH, SENSOR_META_PATH,
]
MISSING_MODEL_ARTIFACTS = [
    path for path in REQUIRED_MODEL_ARTIFACTS if not os.path.exists(path)
]
if MISSING_MODEL_ARTIFACTS:
    raise FileNotFoundError(
        'Missing required model artifacts:\n'
        + '\n'.join(MISSING_MODEL_ARTIFACTS)
    )

with open(CSI_META_PATH, 'r', encoding='utf-8') as handle:
    CSI_META = json.load(handle)
with open(SENSOR_META_PATH, 'r', encoding='utf-8') as handle:
    SENSOR_META = json.load(handle)

if CSI_META.get('classes') != SEVEN_CLASSES:
    raise ValueError(
        f'CSI metadata classes do not match {SEVEN_CLASSES}: '
        f'{CSI_META.get("classes")}'
    )
if SENSOR_META.get('classes') != SENSOR_CLASSES:
    raise ValueError(
        f'Sensor metadata classes do not match {SENSOR_CLASSES}: '
        f'{SENSOR_META.get("classes")}'
    )
if list(CSI_META.get('model_input_shape', [])) != list(EXPECTED_CSI_INPUT_SHAPE):
    warnings.warn(
        'CSI metadata model_input_shape differs from the expected '
        f'{EXPECTED_CSI_INPUT_SHAPE}: {CSI_META.get("model_input_shape")}'
    )
if not (SENSOR_META.get('threshold') or {}).get('value'):
    raise ValueError(
        'smoke_svm.meta.json has no threshold.value; sync/regenerate the '
        'sensor model artifacts before running fusion.'
    )

CSI_FIXTURE_AVAILABLE = (
    os.path.exists(CSI_FIXTURE_PATH) and os.path.exists(CSI_FIXTURE_META_PATH)
)
SENSOR_FIXTURE_AVAILABLE = (
    os.path.exists(SENSOR_FIXTURE_PATH) and os.path.exists(SENSOR_FIXTURE_META_PATH)
)

FIXTURE_FRESHNESS_FAILURES = {}


def _sensor_fixture_freshness():
    if not SENSOR_FIXTURE_AVAILABLE:
        return ['fixture pair missing']
    with open(SENSOR_FIXTURE_META_PATH, 'r', encoding='utf-8') as handle:
        fixture_meta = json.load(handle)
    expected = {
        'pipeline_sha256': sha256_file(SENSOR_PIPELINE_PATH),
        'sensor_meta_sha256': sha256_file(SENSOR_META_PATH),
        'threshold': float(SENSOR_META['threshold']['value']),
        'window_seconds': float(SENSOR_META['preprocessing']['window_seconds']),
    }
    problems = []
    for key, expected_value in expected.items():
        actual = fixture_meta.get(key)
        if isinstance(expected_value, float):
            try:
                actual_float = float(actual)
            except (TypeError, ValueError):
                actual_float = None
            if actual_float is None or abs(actual_float - expected_value) > 1e-9:
                problems.append(
                    f'{key}: fixture={actual!r} expected={expected_value!r}'
                )
        elif actual != expected_value:
            problems.append(
                f'{key}: fixture={actual!r} expected={expected_value!r}'
            )
    if fixture_meta.get('classes') != SENSOR_CLASSES:
        problems.append(
            f'classes: fixture={fixture_meta.get("classes")!r} '
            f'expected={SENSOR_CLASSES!r}'
        )
    if len(fixture_meta.get('interval_keys', [])) != len(
            fixture_meta.get('session_keys', [])):
        problems.append('interval_keys/session_keys length mismatch')
    return problems


def _csi_fixture_freshness():
    if not CSI_FIXTURE_AVAILABLE:
        return ['fixture pair missing']
    with open(CSI_FIXTURE_META_PATH, 'r', encoding='utf-8') as handle:
        fixture_meta = json.load(handle)
    expected = {
        'model_sha256': sha256_file(CSI_MODEL_PATH),
        'platt_sha256': sha256_file(CSI_PLATT_PATH),
        'frozen_split_sha256': CSI_META.get('frozen_split_sha256'),
    }
    problems = []
    for key, expected_value in expected.items():
        actual = fixture_meta.get(key)
        if actual != expected_value:
            problems.append(
                f'{key}: fixture={actual!r} expected={expected_value!r}'
            )
    if fixture_meta.get('classes') != SEVEN_CLASSES:
        problems.append(
            f'classes: fixture={fixture_meta.get("classes")!r} '
            f'expected={SEVEN_CLASSES!r}'
        )
    segment_keys = fixture_meta.get('segment_keys')
    shape = fixture_meta.get('x_test_shape')
    if segment_keys is None:
        problems.append('segment_keys missing from sidecar')
    elif shape and len(segment_keys) != int(shape[0]):
        problems.append(
            f'segment_keys length {len(segment_keys)} != x_test_shape[0] '
            f'{shape[0]}'
        )
    return problems


_sensor_problems = _sensor_fixture_freshness()
if _sensor_problems:
    SENSOR_FIXTURE_AVAILABLE = False
    FIXTURE_FRESHNESS_FAILURES['sensor'] = _sensor_problems
_csi_problems = _csi_fixture_freshness()
if _csi_problems:
    CSI_FIXTURE_AVAILABLE = False
    FIXTURE_FRESHNESS_FAILURES['csi'] = _csi_problems

EXPORT_INSTRUCTIONS = """
[preflight] Test fixtures are missing or stale. To regenerate them:
  1. Open SmokeGuard_Smoke_Sensor.ipynb, run the data-preparation cells and
     the model-load cell (it must load, not retrain), then run the final
     'Fusion fixture export (sensor)' cell.
  2. Open SmokeGuard_v2.ipynb, run the data-preparation cells and the
     model-load cell (FORCE_RETRAIN=False; it must load), then run the final
     'Fusion fixture export (CSI)' cell.
  3. Rerun this fusion notebook from the preflight cell onward.
Fixtures are written to:
  {fixture_dir}
""".format(fixture_dir=FIXTURE_DIR)

if CSI_FIXTURE_AVAILABLE and SENSOR_FIXTURE_AVAILABLE:
    print('[preflight] Both held-out test fixtures are present and fresh.')
else:
    print(EXPORT_INSTRUCTIONS)
    for kind, problems in FIXTURE_FRESHNESS_FAILURES.items():
        print(f'[preflight] {kind} fixture freshness failures:')
        for problem in problems:
            print('  -', problem)

print('[preflight] CSI metadata:')
print(json.dumps({
    key: CSI_META.get(key) for key in (
        'tag', 'classes', 'model_path', 'model_input_shape',
        'frozen_split_sha256', 'smoking_threshold_policy',
    )
}, indent=2))
print('[preflight] Sensor metadata:')
print(json.dumps({
    'tag': SENSOR_META.get('tag'),
    'classes': SENSOR_META.get('classes'),
    'model_path': SENSOR_META.get('model_path'),
    'threshold': SENSOR_META.get('threshold'),
    'feature_count': len(
        SENSOR_META.get('feature_config', {}).get('feature_names', [])
    ),
    'window_seconds': SENSOR_META.get('preprocessing', {}).get('window_seconds'),
    'baseline_seconds': SENSOR_META.get('preprocessing', {}).get('baseline_seconds'),
}, indent=2))


,artifact,path,exists,size_bytes,modified_utc,sha256
0,CSI model,/content/drive/MyDrive/CSI_TRAINING/models/csi_resnet_lstm_7class.keras,True,2769214,2026-10-30T16:08:15Z,42f84bcfdd2c5c23127bf5d6c803e0696ed31fc562fa506ec469ba8240ae4ff7
1,CSI metadata,/content/drive/MyDrive/CSI_TRAINING/models/csi_resnet_lstm_7class.meta.json,True,2130,2026-10-30T16:08:16Z,ea1212d9e0f2323d95981e75edec3e1f31fd3a8895e753ad9b6da10ee63e169d
2,CSI Platt,/content/drive/MyDrive/CSI_TRAINING/models/csi_resnet_lstm_7class.platt.json,True,4005,2026-11-08T15:11:27Z,196f54bbdee83587e659ede2905bfc18990f759e8b77a20a7add74ffb7120547
3,Sensor pipeline,/content/drive/MyDrive/CSI_TRAINING/models/smoke_sensor_svm_v2/smoke_svm_pipeline.joblib,True,23037,2026-11-08T13:37:11Z,221bbe02893e2f4deb203cbbc43098a3cbfe188d4b088eb4f33487dddb58184a
4,Sensor metadata,/content/drive/MyDrive/CSI_TRAINING/models/smoke_sensor_svm_v2/smoke_svm.meta.json,True,4740,2026-11-08T21:56:44Z,2cedce830972a35f25cb31e06bab07f11fec121e3551b52d25d9fe7ddccb7cdb
5,CSI test fixture,/content/drive/MyDrive/CSI_TRAINING/fusion/fixtures/csi_test_fixture.npz,True,16935828,2026-11-08T22:10:16Z,622d0b8be140358e80a40c5cd3e2f2cfd0ff0c71637b6cbf32c2d149c0d5d005
6,CSI fixture metadata,/content/drive/MyDrive/CSI_TRAINING/fusion/fixtures/csi_test_fixture.meta.json,True,16140,2026-11-08T22:10:15Z,32ccd7dcdcbfdfe5a9843becf5b636ebb8d88c205d4061e14b55a3fd49a38043
7,Sensor test fixture,/content/drive/MyDrive/CSI_TRAINING/fusion/fixtures/sensor_test_fixture.npz,True,21486,2026-11-08T21:57:15Z,5aa53c883b3b2fb1a37ecb857c9757395ecf0388fdf7a0e6f51bfe52cf003d76
8,Sensor fixture metadata,/content/drive/MyDrive/CSI_TRAINING/fusion/fixtures/sensor_test_fixture.meta.json,True,14528,2026-11-08T22:12:34Z,54048570500dd432dbf4cad43c5bbe7c9293419ab23f0beb9d19e2fd37ac3487


[preflight] Both held-out test fixtures are present and fresh.
[preflight] CSI metadata:
{
  "tag": "datestrat_balanced_nearestbg_1dresnet_7class",
  "classes": [
    "smoke",
    "eat",
    "drink",
    "sit",
    "stand",
    "random",
    "static"
  ],
  "model_path": "/content/drive/MyDrive/CSI_TRAINING/models/csi_resnet_lstm_7class.keras",
  "model_input_shape": [
    420,
    52
  ],
  "frozen_split_sha256": "1d980314c26c38da91c98ac308d2479c5ccc668a484a2cfba0ca25f3795e155a",
  "smoking_threshold_policy": "session_quantile"
}
[preflight] Sensor metadata:
{
  "tag": "smoke_svm_rbf_file",
  "classes": [
    "no-smoke",
    "smoke"
  ],
  "model_path": "/content/drive/MyDrive/CSI_TRAINING/models/smoke_sensor_svm_v2/smoke_svm_pipeline.joblib",
  "threshold": {
    "objective": "f1",
    "train_oof_f1_at_threshold": 0.9688149688149689,
    "tuned_on": "train_oof_group_cv",
    "value": 0.4049989698494056
  },
  "feature_count": 22,
  "window_seconds": 14.905367895113585,
  "baseline_se

## 3. Sensor inference adapter

The sensor pipeline is `StandardScaler -> SVC(probability=True)`. It expects **raw**
features (the scaler is inside the pipeline), in the exact order stored in its metadata.
The feature extractor is copied from `SmokeGuard_Smoke_Sensor.ipynb` so the fusion
notebook is self-contained.

In [ ]:
# @title 3. Sensor inference adapter
SENSOR_PIPELINE = joblib.load(SENSOR_PIPELINE_PATH)
SENSOR_PIPELINE_CLASSES = list(SENSOR_PIPELINE.classes_)
SENSOR_SMOKE_INDEX = SENSOR_CLASSES.index('smoke')
if len(SENSOR_PIPELINE_CLASSES) != len(SENSOR_CLASSES):
    raise ValueError(
        f'Sensor pipeline has {len(SENSOR_PIPELINE_CLASSES)} classes; '
        f'expected {len(SENSOR_CLASSES)}.'
    )
if all(isinstance(value, (int, np.integer))
       for value in SENSOR_PIPELINE_CLASSES):
    if [int(value) for value in SENSOR_PIPELINE_CLASSES] != list(
            range(len(SENSOR_CLASSES))):
        raise ValueError(
            'Integer-encoded sensor classes do not match the metadata order: '
            f'{SENSOR_PIPELINE_CLASSES}'
        )
    print('[sensor] pipeline uses integer-encoded classes; mapped to '
          f'{SENSOR_CLASSES} by metadata order.')
else:
    if SENSOR_PIPELINE_CLASSES != SENSOR_CLASSES:
        raise ValueError(
            f'Sensor pipeline classes are {SENSOR_PIPELINE_CLASSES}; '
            f'expected {SENSOR_CLASSES}.'
        )
    SENSOR_SMOKE_INDEX = SENSOR_PIPELINE_CLASSES.index('smoke')

SENSOR_CHANNELS = list(SENSOR_META['feature_config']['channels'])
SENSOR_FEATURE_NAMES = list(SENSOR_META['feature_config']['feature_names'])
SENSOR_DROPPED_COLUMNS = list(SENSOR_META['feature_config']['dropped_columns'])
WINDOW_SECONDS = float(SENSOR_META['preprocessing']['window_seconds'])
BASELINE_SECONDS = float(SENSOR_META['preprocessing']['baseline_seconds'])
BASELINE_CHANNEL = SENSOR_META['preprocessing'].get('baseline_channel', 'gas_kohm')
MIN_SAMPLES_PER_INTERVAL = int(
    SENSOR_META['preprocessing']['min_samples_per_interval']
)
MIN_BASELINE_SAMPLES = int(
    SENSOR_META['preprocessing']['min_baseline_samples']
)
_threshold_block = SENSOR_META.get('threshold') or {}
SENSOR_THRESHOLD = _threshold_block.get('value')
if SENSOR_THRESHOLD is None:
    raise ValueError(
        'smoke_svm.meta.json has no threshold.value. Sync the retrained sensor '
        'model metadata; the fusion policy must not fall back to a stale gate.'
    )
SENSOR_THRESHOLD = float(SENSOR_THRESHOLD)
FEATURE_NAMES = SENSOR_FEATURE_NAMES  # alias used by the copied extractor

ALL_FEATURE_NAMES = []
for channel in SENSOR_CHANNELS:
    ALL_FEATURE_NAMES.extend([
        f'{channel}__base_diff',
        f'{channel}__std',
        f'{channel}__first_last_diff',
    ])
ALL_FEATURE_NAMES.append(f'{BASELINE_CHANNEL}__base_ratio')
EXPECTED_KEPT_FEATURES = [
    name for name in ALL_FEATURE_NAMES
    if name not in set(SENSOR_DROPPED_COLUMNS)
]
if SENSOR_FEATURE_NAMES != EXPECTED_KEPT_FEATURES:
    raise ValueError(
        'Sensor feature order does not match the copied extractor. '
        f'Metadata: {SENSOR_FEATURE_NAMES}\nExpected: {EXPECTED_KEPT_FEATURES}'
    )


def _make_feature_row(window_df, baseline_df):
    """Session-robust features for one inference window (copied from the sensor notebook)."""
    feature_row = {}
    for channel in SENSOR_CHANNELS:
        series = window_df[channel].to_numpy(dtype=np.float64)
        baseline_values = baseline_df[channel].to_numpy(dtype=np.float64)
        baseline_median = float(np.median(baseline_values))
        feature_row[f'{channel}__base_diff'] = (
            float(np.mean(series)) - baseline_median
        )
        feature_row[f'{channel}__std'] = float(np.std(series, ddof=0))
        feature_row[f'{channel}__first_last_diff'] = (
            float(series[-1] - series[0])
        )

    window_gas_mean = float(
        np.mean(window_df[BASELINE_CHANNEL].to_numpy(dtype=np.float64))
    )
    gas_baseline_median = float(
        np.median(baseline_df[BASELINE_CHANNEL].to_numpy(dtype=np.float64))
    )
    feature_row[f'{BASELINE_CHANNEL}__base_ratio'] = (
        window_gas_mean / max(gas_baseline_median, 1e-9)
    )
    return feature_row


def _sensor_window_slices(samples_df, window_seconds=None, baseline_seconds=None):
    """Causal window/baseline slices with explicit nanosecond time validation."""
    if 'time' not in samples_df.columns:
        raise ValueError('Sensor samples must include a time column (nanoseconds).')
    if len(samples_df) == 0:
        raise ValueError('Sensor sample buffer is empty.')
    time_values = samples_df['time'].to_numpy()
    if time_values.dtype.kind not in 'iu':
        raise ValueError(
            'Sensor time must be integer nanoseconds. Convert the buffer '
            'explicitly; float seconds/microseconds are rejected.'
        )
    if int(time_values.max()) < 10 ** 15:
        raise ValueError(
            'Sensor time values look like seconds or microseconds; expected '
            'nanosecond epoch values.'
        )

    window_seconds = (
        WINDOW_SECONDS if window_seconds is None else float(window_seconds)
    )
    baseline_seconds = (
        BASELINE_SECONDS if baseline_seconds is None else float(baseline_seconds)
    )
    times = time_values.astype('int64').astype(np.float64) / 1e9
    order = np.argsort(times, kind='stable')
    samples_df = samples_df.iloc[order].reset_index(drop=True)
    times = times[order]

    end_time = times[-1]
    window_start = end_time - window_seconds
    window_rows = samples_df.loc[times >= window_start]
    baseline_rows = samples_df.loc[
        (times >= window_start - baseline_seconds) & (times < window_start)
    ]

    if len(window_rows) < MIN_SAMPLES_PER_INTERVAL:
        raise ValueError(
            f'Only {len(window_rows)} sensor sample(s) in the last '
            f'{window_seconds:.3f} s; at least {MIN_SAMPLES_PER_INTERVAL} '
            'are required.'
        )
    if len(baseline_rows) < MIN_BASELINE_SAMPLES:
        raise ValueError(
            f'Only {len(baseline_rows)} baseline sample(s) before the window; '
            f'at least {MIN_BASELINE_SAMPLES} are required.'
        )
    return window_rows, baseline_rows


@dataclass(frozen=True)
class SensorPrediction:
    p_smoke: float
    threshold: float
    hit: bool
    n_window_samples: int
    n_baseline_samples: int

    def to_dict(self) -> dict:
        return asdict(self)


def extract_features_for_inference(samples_df, window_seconds=None,
                                   baseline_seconds=None):
    """Reference extractor for the late-fusion layer (raw feature matrix)."""
    window_rows, baseline_rows = _sensor_window_slices(
        samples_df, window_seconds, baseline_seconds
    )
    feature_row = _make_feature_row(window_rows, baseline_rows)
    return np.array(
        [[feature_row[name] for name in FEATURE_NAMES]], dtype=np.float64
    )


def predict_sensor_smoke(samples_df, window_seconds=None,
                         baseline_seconds=None) -> SensorPrediction:
    window_rows, baseline_rows = _sensor_window_slices(
        samples_df, window_seconds, baseline_seconds
    )
    feature_row = _make_feature_row(window_rows, baseline_rows)
    matrix = np.array(
        [[feature_row[name] for name in FEATURE_NAMES]], dtype=np.float64
    )
    probability = float(
        SENSOR_PIPELINE.predict_proba(matrix)[0, SENSOR_SMOKE_INDEX]
    )
    if not np.isfinite(probability):
        raise RuntimeError('Sensor pipeline returned a non-finite probability.')
    return SensorPrediction(
        p_smoke=probability,
        threshold=SENSOR_THRESHOLD,
        hit=probability >= SENSOR_THRESHOLD,
        n_window_samples=len(window_rows),
        n_baseline_samples=len(baseline_rows),
    )


print(f'[sensor] pipeline: {SENSOR_PIPELINE_PATH}')
print(f'[sensor] positive class: {SENSOR_CLASSES[SENSOR_SMOKE_INDEX]} '
      f'(predict_proba column {SENSOR_SMOKE_INDEX})')
print(f'[sensor] threshold: {SENSOR_THRESHOLD:.12f}')
print(f'[sensor] window: {WINDOW_SECONDS:.6f} s | baseline: '
      f'{BASELINE_SECONDS:.0f} s ({BASELINE_CHANNEL})')
print(f'[sensor] features ({len(SENSOR_FEATURE_NAMES)}): '
      f'{SENSOR_FEATURE_NAMES}')
print(f'[sensor] dropped columns: {SENSOR_DROPPED_COLUMNS}')


[sensor] pipeline uses integer-encoded classes; mapped to ['no-smoke', 'smoke'] by metadata order.
[sensor] pipeline: /content/drive/MyDrive/CSI_TRAINING/models/smoke_sensor_svm_v2/smoke_svm_pipeline.joblib
[sensor] positive class: smoke (predict_proba column 1)
[sensor] threshold: 0.404998969849
[sensor] window: 14.905368 s | baseline: 600 s (gas_kohm)
[sensor] features (22): ['temp_c__base_diff', 'temp_c__std', 'temp_c__first_last_diff', 'humidity_pct__base_diff', 'humidity_pct__std', 'humidity_pct__first_last_diff', 'pressure_hpa__base_diff', 'pressure_hpa__std', 'pressure_hpa__first_last_diff', 'gas_kohm__base_diff', 'gas_kohm__std', 'gas_kohm__first_last_diff', 'pm1_0__base_diff', 'pm1_0__std', 'pm1_0__first_last_diff', 'pm2_5__base_diff', 'pm2_5__std', 'pm2_5__first_last_diff', 'pm10__base_diff', 'pm10__std', 'pm10__first_last_diff', 'gas_kohm__base_ratio']
[sensor] dropped columns: []


## 4. CSI inference adapter

The CSI checkpoint is the seven-class 1D ResNet-LSTM
(`csi_resnet_lstm_7class.keras`). It needs the custom `TimeSeriesAugment`
layer at load time (`compile=False` avoids the custom training loss). Fusion uses:

- **calibrated seven-class argmax** for the gesture gate;
- **binary Platt `P(smoke)`** for the weighted score (the calibrator stored in the
  `csi_resnet_lstm_7class.platt.json` sidecar).

The seven classes are `smoke, eat, drink, sit, stand, random, static`. `static`
is the idle/background class and never counts as a gesture hit. The deployed
smoke threshold (`session_quantile`) is not reused as a fusion gate; the
calibrated argmax is the CSI gate.


In [ ]:
# @title 4. CSI inference adapter
@tf.keras.utils.register_keras_serializable(package='smokeguard')
class TimeSeriesAugment(layers.Layer):
    """Training-only 1D augmentation (copied from the CSI notebook)."""

    def __init__(self, shift_max=12, scale_min=0.9, scale_max=1.1,
                 noise_std=0.05, **kwargs):
        super().__init__(**kwargs)
        self.shift_max = shift_max
        self.scale_min = scale_min
        self.scale_max = scale_max
        self.noise_std = noise_std

    def call(self, x, training=False):
        if not training:
            return x
        batch = tf.shape(x)[0]
        scale = tf.random.uniform(
            [batch, 1, 1], self.scale_min, self.scale_max
        )
        x = x * scale
        std = tf.math.reduce_std(x, axis=[1, 2], keepdims=True)
        x = x + tf.random.normal(tf.shape(x), stddev=self.noise_std) * std
        if self.shift_max > 0:
            shift = tf.random.uniform(
                [], -self.shift_max, self.shift_max + 1, dtype=tf.int32
            )
            x = tf.roll(x, shift=shift, axis=1)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            'shift_max': self.shift_max,
            'scale_min': self.scale_min,
            'scale_max': self.scale_max,
            'noise_std': self.noise_std,
        })
        return config


CSI_MODEL = keras.models.load_model(
    CSI_MODEL_PATH,
    compile=False,
    custom_objects={'TimeSeriesAugment': TimeSeriesAugment},
)
if tuple(CSI_MODEL.input_shape[1:]) != EXPECTED_CSI_INPUT_SHAPE:
    raise ValueError(
        f'CSI model input shape is {CSI_MODEL.input_shape}; expected '
        f'(None, {EXPECTED_CSI_INPUT_SHAPE[0]}, {EXPECTED_CSI_INPUT_SHAPE[1]}).'
    )
activity_layer = CSI_MODEL.get_layer('activity')
activity_width = int(getattr(activity_layer, 'units', -1))
if activity_width != len(SEVEN_CLASSES):
    raise ValueError(
        'CSI activity head width does not match the seven-class list: '
        f'{activity_width} != {len(SEVEN_CLASSES)}.'
    )

with open(CSI_PLATT_USED_PATH, 'r', encoding='utf-8') as handle:
    CSI_PLATT_PAYLOAD = json.load(handle)
if CSI_PLATT_PAYLOAD['multiclass']['classes'] != SEVEN_CLASSES:
    raise ValueError(
        'Platt calibrator classes do not match the CSI seven-class list.'
    )
if CSI_PLATT_PAYLOAD.get('smoke_binary', {}).get('class') != 'smoke':
    raise ValueError('Platt smoke_binary block does not target smoke.')
SMOKE_BINARY_A = float(CSI_PLATT_PAYLOAD['smoke_binary']['slope_A'])
SMOKE_BINARY_B = float(CSI_PLATT_PAYLOAD['smoke_binary']['intercept_B'])
CSI_SMOKE_INDEX = SEVEN_CLASSES.index('smoke')


def _stable_sigmoid(z):
    return np.exp(-np.logaddexp(0.0, -z))


class PlattCalibrator:
    """One-vs-rest Platt scaling (load/transform only; fitting is in the CSI notebook)."""

    def __init__(self, classes, slopes, intercepts):
        self.classes = list(classes)
        self.slopes = np.asarray(slopes, dtype=np.float64)
        self.intercepts = np.asarray(intercepts, dtype=np.float64)

    @classmethod
    def from_dict(cls, payload):
        if payload['classes'] != list(SEVEN_CLASSES):
            raise ValueError(
                'Saved Platt calibrator classes do not match SEVEN_CLASSES.'
            )
        return cls(
            classes=payload['classes'],
            slopes=payload['slope_A'],
            intercepts=payload['intercept_B'],
        )

    def transform(self, logits):
        logits = np.asarray(logits, dtype=np.float64)
        if logits.shape[1] != len(self.classes):
            raise ValueError('Logit width does not match the class list.')
        calibrated = _stable_sigmoid(
            logits * self.slopes[None, :] + self.intercepts[None, :]
        )
        totals = calibrated.sum(axis=1, keepdims=True)
        uniform = np.full_like(calibrated, 1.0 / calibrated.shape[1])
        return np.where(
            totals > 0.0, calibrated / np.maximum(totals, 1e-12), uniform
        )

    def predict_proba(self, model, X):
        return self.transform(get_model_logits(model, X))


CSI_PLATT = PlattCalibrator.from_dict(CSI_PLATT_PAYLOAD['multiclass'])
_FEATURE_MODEL = None


def get_model_logits(model, X):
    """Exact pre-softmax logits: dense_features @ activity kernel + bias."""
    global _FEATURE_MODEL
    layer_names = {layer.name for layer in model.layers}
    if 'dense_features' not in layer_names or 'activity' not in layer_names:
        raise ValueError(
            'CSI model is missing the expected dense_features/activity layers.'
        )
    if _FEATURE_MODEL is None:
        _FEATURE_MODEL = models.Model(
            inputs=model.input,
            outputs=model.get_layer('dense_features').output,
        )
    features = _FEATURE_MODEL.predict(X, verbose=0)
    kernel, bias = model.get_layer('activity').get_weights()
    if tuple(kernel.shape) != (int(features.shape[-1]), len(SEVEN_CLASSES)):
        raise ValueError(
            f'Activity kernel shape {kernel.shape} does not match '
            f'({features.shape[-1]}, {len(SEVEN_CLASSES)}).'
        )
    if tuple(bias.shape) != (len(SEVEN_CLASSES),):
        raise ValueError(
            f'Activity bias shape {bias.shape} does not match '
            f'({len(SEVEN_CLASSES)},).'
        )
    return features @ kernel + bias


@dataclass(frozen=True)
class CsiOutput:
    prob_cal: np.ndarray
    prob_raw: np.ndarray
    p_smoke_multiclass_cal: np.ndarray
    p_smoke_binary_cal: np.ndarray
    argmax_cal: np.ndarray
    argmax_raw: np.ndarray

    def __len__(self) -> int:
        return int(self.prob_cal.shape[0])


def predict_csi_proba(X, background_subtracted=True) -> CsiOutput:
    """Return calibrated/raw seven-class probabilities for background-subtracted windows."""
    if not background_subtracted:
        raise ValueError(
            'CSI inference requires background-subtracted windows. Pass the '
            '(420, 52) window after static-background subtraction.'
        )
    array = np.asarray(X, dtype=np.float32)
    single = array.ndim == 2
    if single:
        array = array[None, ...]
    if array.ndim != 3 or tuple(array.shape[1:]) != EXPECTED_CSI_INPUT_SHAPE:
        raise ValueError(
            f'CSI input must have shape (N, 420, 52); got {array.shape}.'
        )
    if not np.all(np.isfinite(array)):
        raise ValueError('CSI input contains NaN or infinite values.')

    logits = get_model_logits(CSI_MODEL, array)
    if not np.all(np.isfinite(logits)):
        raise ValueError('CSI model returned non-finite logits.')
    prob_cal = CSI_PLATT.transform(logits)
    prob_raw = tf.nn.softmax(logits, axis=1).numpy()
    p_smoke_multi = prob_cal[:, CSI_SMOKE_INDEX]
    p_smoke_binary = _stable_sigmoid(
        SMOKE_BINARY_A * logits[:, CSI_SMOKE_INDEX] + SMOKE_BINARY_B
    )
    argmax_cal = np.array(
        [SEVEN_CLASSES[int(index)] for index in prob_cal.argmax(axis=1)],
        dtype=object,
    )
    argmax_raw = np.array(
        [SEVEN_CLASSES[int(index)] for index in prob_raw.argmax(axis=1)],
        dtype=object,
    )
    return CsiOutput(
        prob_cal=prob_cal,
        prob_raw=prob_raw,
        p_smoke_multiclass_cal=p_smoke_multi,
        p_smoke_binary_cal=p_smoke_binary,
        argmax_cal=argmax_cal,
        argmax_raw=argmax_raw,
    )


print(f'[csi] model: {CSI_MODEL_PATH}')
print(f'[csi] input shape: {CSI_MODEL.input_shape}')
print(f'[csi] classes: {SEVEN_CLASSES}')
print(f'[csi] Platt calibrator: {CSI_PLATT_USED_PATH}')
print(f'[csi] binary smoke Platt: A={SMOKE_BINARY_A:.6f}, '
      f'B={SMOKE_BINARY_B:.6f}')


[csi] model: /content/drive/MyDrive/CSI_TRAINING/models/csi_resnet_lstm_7class.keras
[csi] input shape: (None, 420, 52)
[csi] classes: ['smoke', 'eat', 'drink', 'sit', 'stand', 'random', 'static']
[csi] Platt calibrator: /content/drive/MyDrive/CSI_TRAINING/models/csi_resnet_lstm_7class.platt.json
[csi] binary smoke Platt: A=1.465929, B=-3.731353


## Raw CSI preprocessing

The fusion core accepts a **background-subtracted `(420, 52)` window**. The exact raw
preprocessing chain (Hampel filter, timestamp resampling to 420 packets, Butterworth
low-pass, Gaussian smoothing, and same-date nearest-static background subtraction) lives
in `SmokeGuard_v2.ipynb` cells 7–9 (`preprocess_csi_data` and
`select_nearest_static_backgrounds`). It is intentionally not duplicated here; the test
fixtures already contain background-subtracted held-out windows, and a live deployment
should call the CSI notebook's preprocessing before this adapter.

## 5. Fusion policy (deployed backend contract)

The policy is a frozen dataclass with hard validation. The deployed policy uses
**balanced 0.50 CSI / 0.50 sensor weights** and a fused threshold of **0.70**.
The safety rule is the conjunction for a **`fusion`** alert: both the CSI
gesture gate and the sensor gate must be on, and the weighted score must clear
0.70. A sensor gate on its own produces **`smoke_only`** — a possible
sensor-only alert. The experimental **`csi_only`** class (off by default) covers
a CSI gesture when the sensor operates below its gate; it has no fused score.

The backend's `FusionPolicy` also carries `csi_only_enabled` and
`csi_only_min_prob`; these are serialized in the persisted policy so the
notebook and the deployment cannot drift silently.


In [ ]:
# @title 5. Fusion policy (deployed backend contract)
COMPARISON_EPS = 1e-12
CSI_PROB_ANCHOR = 0.50


@dataclass(frozen=True)
class FusionPolicy:
    weight_csi: float = FUSION_CSI_WEIGHT
    weight_sensor: float = FUSION_SENSOR_WEIGHT
    csi_gate_mode: str = 'argmax'  # 'argmax' or 'probability'
    csi_gate_threshold: float = 0.50
    sensor_gate_threshold: float = SENSOR_THRESHOLD
    fused_threshold: float = FUSION_SCORE_THRESHOLD
    require_both: bool = True
    allow_sub_anchor: bool = False
    csi_only_enabled: bool = CSI_ONLY_ENABLED
    csi_only_min_prob: float = CSI_ONLY_MIN_PROB
    schema_version: int = FUSION_SCHEMA_VERSION

    def __post_init__(self):
        if self.csi_gate_mode not in ('argmax', 'probability'):
            raise ValueError(
                "csi_gate_mode must be 'argmax' or 'probability'."
            )
        for name in ('csi_gate_threshold', 'sensor_gate_threshold',
                     'fused_threshold', 'csi_only_min_prob'):
            value = float(getattr(self, name))
            if not 0.0 <= value <= 1.0:
                raise ValueError(f'{name} must be in [0, 1]; got {value}.')
        for name in ('weight_csi', 'weight_sensor'):
            value = float(getattr(self, name))
            if not 0.0 <= value <= 1.0:
                raise ValueError(f'{name} must be in [0, 1]; got {value}.')
        total = float(self.weight_csi) + float(self.weight_sensor)
        if abs(total - 1.0) > 1e-9:
            raise ValueError(
                f'Fusion weights must sum to 1.0; got {total:.12f}.'
            )
        if not self.require_both:
            raise ValueError(
                'The deployed policy requires the CSI gesture gate and the '
                'sensor gate for a confirmed fusion alert; require_both must '
                'remain True.'
            )
        if (self.csi_gate_mode == 'probability'
                and not self.allow_sub_anchor
                and self.fused_threshold
                <= self.anchor_score() + COMPARISON_EPS):
            warnings.warn(
                'fused_threshold is at or below the weighted probability-gate '
                f'anchor ({self.anchor_score():.6f}); the weighted score will '
                'not add a decision constraint. Set allow_sub_anchor=True to '
                'silence this warning.',
                UserWarning,
                stacklevel=2,
            )

    def anchor_score(self) -> float:
        """Score when both probability gates are exactly at threshold."""
        return (float(self.weight_sensor) * float(self.sensor_gate_threshold)
                + float(self.weight_csi) * float(CSI_PROB_ANCHOR))

    def to_dict(self) -> dict:
        payload = asdict(self)
        payload['anchor_score'] = self.anchor_score()
        return payload

    @classmethod
    def from_dict(cls, payload: dict) -> 'FusionPolicy':
        fields = {
            'weight_csi', 'weight_sensor', 'csi_gate_mode',
            'csi_gate_threshold', 'sensor_gate_threshold',
            'fused_threshold', 'require_both', 'allow_sub_anchor',
            'csi_only_enabled', 'csi_only_min_prob', 'schema_version',
        }
        return cls(**{key: payload[key] for key in fields if key in payload})


DEFAULT_POLICY = FusionPolicy()
print('[fusion] policy (backend ground truth):')
for key, value in DEFAULT_POLICY.to_dict().items():
    print(f'  {key}: {value}')


@dataclass(frozen=True)
class FusionDecision:
    label: str | None          # 'fusion' | 'smoke_only' | 'csi_only' | None
    alert: bool                # any of the three labels fires a WS alert
    reason: str
    fused_score: float | None  # None for csi_only / no event
    csi_hit: bool
    sensor_hit: bool
    p_csi_smoke: float
    p_sensor_smoke: float
    csi_gate_mode: str

    @property
    def alarm(self) -> bool:
        """Confirmed fusion alert (kept for backwards-compatible tables)."""
        return self.label == 'fusion'

    def to_dict(self) -> dict:
        payload = asdict(self)
        payload['alarm'] = self.alarm
        return payload


[fusion] policy (backend ground truth):
  weight_csi: 0.5
  weight_sensor: 0.5
  csi_gate_mode: argmax
  csi_gate_threshold: 0.5
  sensor_gate_threshold: 0.4049989698494056
  fused_threshold: 0.7
  require_both: True
  allow_sub_anchor: False
  csi_only_enabled: False
  csi_only_min_prob: 0.0
  schema_version: 2
  anchor_score: 0.4524994849247028


## 6. Fusion functions

`fuse_scores` is vectorized for batch evaluation; `decide` is the scalar
interface with a human-readable reason code. Probabilities are validated rather
than silently clipped.

The functions mirror `app/inference.fuse()`: they return the deployed
`label` (`fusion`, `smoke_only`, `csi_only`, or null), an `alert` boolean
(true for all three labels), a nullable `fused_score`, and the diagnostic
`reason`. `alarm` remains an alias for `label == 'fusion'` so older tables
still work.


In [ ]:
# @title 6. Fusion functions (mirror of app/inference.fuse)
def _as_1d_probability(name, values):
    array = np.asarray(values, dtype=np.float64)
    if array.ndim != 1:
        raise ValueError(f'{name} must be one-dimensional; got {array.shape}.')
    if not np.all(np.isfinite(array)):
        raise ValueError(f'{name} contains NaN or infinite values.')
    if np.any(array < 0.0) or np.any(array > 1.0):
        raise ValueError(f'{name} must be in [0, 1].')
    return array


def assign_csi_hit(p_csi_smoke, csi_argmax_cal, policy=DEFAULT_POLICY):
    """CSI gesture gate from calibrated argmax or calibrated probability."""
    if policy.csi_gate_mode == 'argmax':
        argmax = np.asarray(csi_argmax_cal)
        if argmax.ndim != 1:
            raise ValueError('csi_argmax_cal must be one-dimensional.')
        return argmax == 'smoke'
    probabilities = _as_1d_probability('p_csi_smoke', p_csi_smoke)
    return probabilities >= policy.csi_gate_threshold


def fuse_scores(p_csi_smoke, p_sensor_smoke, csi_hit,
                policy=DEFAULT_POLICY) -> pd.DataFrame:
    """Vectorised deployed policy with the backend's alert labels.

    label:
      fusion      csi_hit AND sensor_hit AND score >= fused_threshold
      smoke_only  sensor_hit but not a fusion confirmation (possible alert)
      csi_only    experimental: csi_hit with the sensor operating below its
                  gate; fused_score is None (no fusion was computed)
      None        no alert
    """
    p_csi = _as_1d_probability('p_csi_smoke', p_csi_smoke)
    p_sensor = _as_1d_probability('p_sensor_smoke', p_sensor_smoke)
    csi_gate = np.asarray(csi_hit, dtype=bool)
    if p_csi.shape != p_sensor.shape or p_csi.shape != csi_gate.shape:
        raise ValueError(
            'p_csi_smoke, p_sensor_smoke, and csi_hit must have the same '
            f'shape; got {p_csi.shape}, {p_sensor.shape}, {csi_gate.shape}.'
        )

    score = (float(policy.weight_csi) * p_csi
             + float(policy.weight_sensor) * p_sensor)
    sensor_gate = p_sensor >= float(policy.sensor_gate_threshold)
    above_fused = score >= float(policy.fused_threshold) - COMPARISON_EPS

    fusion = csi_gate & sensor_gate & above_fused
    csi_only = (
        bool(policy.csi_only_enabled)
        & ~sensor_gate
        & csi_gate
        & (p_csi >= float(policy.csi_only_min_prob))
    )

    label = np.select(
        [fusion, csi_only, sensor_gate],
        ['fusion', 'csi_only', 'smoke_only'],
        default=None,
    )
    alert = np.array([value is not None for value in label], dtype=bool)
    alarm = label == 'fusion'
    fused_score = np.where(
        np.isin(label, ['fusion', 'smoke_only']), score, np.nan
    )
    reason = np.select(
        [fusion, csi_only, sensor_gate & ~csi_gate,
         sensor_gate & csi_gate],
        ['alarm', 'csi_only', 'no_csi_gesture', 'below_fused_threshold'],
        default='no_sensor_smoke',
    )

    return pd.DataFrame({
        'p_csi_smoke': p_csi,
        'p_sensor_smoke': p_sensor,
        'csi_hit': csi_gate,
        'sensor_hit': sensor_gate,
        'fused_score': fused_score,
        'label': label,
        'alert': alert,
        'alarm': alarm,
        'reason': reason,
    })


def decide(p_csi_smoke, p_sensor_smoke, csi_hit,
           policy=DEFAULT_POLICY) -> FusionDecision:
    """Scalar wrapper around fuse_scores with a convenient decision object."""
    table = fuse_scores(
        np.array([p_csi_smoke], dtype=np.float64),
        np.array([p_sensor_smoke], dtype=np.float64),
        np.array([csi_hit], dtype=bool),
        policy,
    ).iloc[0]
    raw_label = table['label']
    label = None if raw_label is None or pd.isna(raw_label) else str(raw_label)
    fused_score = (
        None if pd.isna(table['fused_score']) else float(table['fused_score'])
    )
    return FusionDecision(
        label=label,
        alert=bool(table['alert']),
        reason=str(table['reason']),
        fused_score=fused_score,
        csi_hit=bool(table['csi_hit']),
        sensor_hit=bool(table['sensor_hit']),
        p_csi_smoke=float(table['p_csi_smoke']),
        p_sensor_smoke=float(table['p_sensor_smoke']),
        csi_gate_mode=policy.csi_gate_mode,
    )


def fuse_models(csi_window_bg, sensor_samples_df, policy=DEFAULT_POLICY):
    """End-to-end convenience: one background-subtracted CSI window + sensor buffer."""
    csi_output = predict_csi_proba(csi_window_bg, background_subtracted=True)
    if len(csi_output) != 1:
        raise ValueError('fuse_models expects exactly one CSI window.')
    sensor_output = predict_sensor_smoke(sensor_samples_df)
    csi_hit = bool(assign_csi_hit(
        csi_output.p_smoke_binary_cal,
        csi_output.argmax_cal,
        policy,
    )[0])
    decision = decide(
        float(csi_output.p_smoke_binary_cal[0]),
        float(sensor_output.p_smoke),
        csi_hit,
        policy,
    )
    return {
        'csi': csi_output,
        'sensor': sensor_output,
        'decision': decision,
    }


print('[fusion] fuse_scores / decide / fuse_models are ready.')


[fusion] fuse_scores / decide / fuse_models are ready.


## 7. Synthetic rule tests

These deterministic cases prove the deployed policy rules before any test-set
simulation. They cover the three alert labels (`fusion`, `smoke_only`,
`csi_only`), the csi-only toggle, sensor misses, CSI gate misses, boundary
scores around the 0.70 fused threshold, invalid inputs, and policy
serialization. The expected outcomes use the backend's labels, not the old
AND-only "alarm" boolean.


In [ ]:
# @title 7. Synthetic rule tests (deployed labels)
def _rule_suite(policy, cases, policy_name):
    rows = []
    for name, p_csi, p_sensor, csi_hit, expected_label in cases:
        decision = decide(p_csi, p_sensor, csi_hit, policy)
        rows.append({
            'policy': policy_name,
            'case': name,
            'p_csi_smoke': p_csi,
            'p_sensor_smoke': p_sensor,
            'csi_hit': csi_hit,
            'sensor_hit': decision.sensor_hit,
            'fused_score': decision.fused_score,
            'label': decision.label,
            'alert': decision.alert,
            'alarm': decision.alarm,
            'reason': decision.reason,
            'expected_label': expected_label,
            'expected_alert': expected_label is not None,
            'passed': (decision.label == expected_label
                       and decision.alert == (expected_label is not None)),
        })
    return pd.DataFrame(rows)


DEFAULT_RULE_CASES = [
    ('sensor_only_smoke_no_csi_gesture', 0.05, 0.99, False, 'smoke_only'),
    ('csi_only_gesture_sensor_silent', 0.95, 0.05, True, None),
    ('true_smoking_sensor_missed', 0.90, 0.20, True, None),
    ('drink_misclassified_sensor_silent', 0.80, 0.20, True, None),
    ('both_strong_alarm', 0.95, 0.99, True, 'fusion'),
    ('both_gates_weak_csi_sensor_at_threshold',
     0.10, DEFAULT_POLICY.sensor_gate_threshold, True, 'smoke_only'),
    ('sensor_compensates_moderate_csi', 0.50, 0.95, True, 'fusion'),
    ('sensor_at_exact_gate',
     0.90, DEFAULT_POLICY.sensor_gate_threshold, True, 'smoke_only'),
    ('score_just_above_fused_threshold', 0.61, 0.80, True, 'fusion'),
    ('score_just_below_fused_threshold', 0.59, 0.80, True, 'smoke_only'),
    ('csi_gate_false_overrides_high_sensor', 0.02, 1.00, False, 'smoke_only'),
    ('sensor_gate_false_overrides_high_csi',
     1.00, DEFAULT_POLICY.sensor_gate_threshold - 0.05, True, None),
]

CSI_ONLY_POLICY = replace(DEFAULT_POLICY, csi_only_enabled=True)
CSI_ONLY_RULE_CASES = [
    ('csi_only_enabled_gesture_sensor_silent', 0.95, 0.05, True, 'csi_only'),
    ('csi_only_enabled_sensor_at_gate',
     0.95, CSI_ONLY_POLICY.sensor_gate_threshold, True, 'smoke_only'),
    ('csi_only_enabled_zero_prob_allowed', 0.0, 0.05, True, 'csi_only'),
]

RULE_RESULTS = pd.concat([
    _rule_suite(DEFAULT_POLICY, DEFAULT_RULE_CASES, 'default'),
    _rule_suite(CSI_ONLY_POLICY, CSI_ONLY_RULE_CASES, 'csi_only_enabled'),
], ignore_index=True)
display(RULE_RESULTS)
assert RULE_RESULTS['passed'].all(), 'Synthetic rule suite failed.'

# Invalid probability inputs.
for bad in (np.nan, -0.1, 1.1):
    for argument in ('csi', 'sensor'):
        try:
            if argument == 'csi':
                decide(bad, 0.9, True, DEFAULT_POLICY)
            else:
                decide(0.9, bad, True, DEFAULT_POLICY)
        except ValueError:
            pass
        else:
            raise AssertionError(f'Invalid {argument} probability {bad} accepted.')

# Invalid policy configurations.
try:
    FusionPolicy(weight_csi=0.4, weight_sensor=0.4)
except ValueError:
    pass
else:
    raise AssertionError('Weights that do not sum to 1 were accepted.')
try:
    FusionPolicy(weight_csi=0.6, weight_sensor=0.6)
except ValueError:
    pass
else:
    raise AssertionError('Weights that do not sum to 1 were accepted.')
try:
    FusionPolicy(csi_only_min_prob=1.1)
except ValueError:
    pass
else:
    raise AssertionError('csi_only_min_prob outside [0, 1] was accepted.')
try:
    FusionPolicy(require_both=False)
except ValueError:
    pass
else:
    raise AssertionError('require_both=False was accepted.')

# Equal weights are the deployed policy and must be accepted.
assert FusionPolicy(weight_csi=0.5, weight_sensor=0.5).weight_sensor == 0.5
assert DEFAULT_POLICY.weight_csi == DEFAULT_POLICY.weight_sensor == 0.5

# Sub-anchor warning in probability mode (anchor is 0.4525 with 50/50).
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    FusionPolicy(csi_gate_mode='probability', fused_threshold=0.45)
assert any(issubclass(w.category, UserWarning) for w in caught), (
    'Sub-anchor fused threshold did not warn.'
)

# JSON round-trip and vectorised/scalar agreement.
encoded = json.loads(json.dumps(DEFAULT_POLICY.to_dict()))
assert FusionPolicy.from_dict(encoded) == DEFAULT_POLICY
rng = np.random.default_rng(42)
p_c = rng.random(100)
p_s = rng.random(100)
hits = rng.random(100) > 0.5
batch = fuse_scores(p_c, p_s, hits, DEFAULT_POLICY)
for index in range(100):
    scalar = decide(float(p_c[index]), float(p_s[index]),
                    bool(hits[index]), DEFAULT_POLICY)
    row = batch.iloc[index]
    assert bool(row['alert']) == scalar.alert
    assert row['label'] == scalar.label
    assert row['reason'] == scalar.reason
    assert (pd.isna(row['fused_score'])
            == (scalar.fused_score is None))
    if scalar.fused_score is not None:
        assert abs(float(row['fused_score']) - scalar.fused_score) < 1e-12

# Gate-mode behavior.
PROB_POLICY_CHECK = FusionPolicy(
    csi_gate_mode='probability', csi_gate_threshold=0.5,
    fused_threshold=0.70,
)
assert assign_csi_hit(
    np.array([0.49, 0.50, 0.51]),
    np.array(['smoke', 'smoke', 'smoke']),
    PROB_POLICY_CHECK,
).tolist() == [False, True, True]
assert assign_csi_hit(
    np.array([0.9, 0.05]),
    np.array(['smoke', 'drink']),
    DEFAULT_POLICY,
).tolist() == [True, False]

print('\n[pure-engine] all synthetic rule and policy checks passed.')
print(f'[pure-engine] anchor score: {DEFAULT_POLICY.anchor_score():.6f}')


,policy,case,p_csi_smoke,p_sensor_smoke,csi_hit,sensor_hit,fused_score,label,alert,alarm,reason,expected_label,expected_alert,passed
0,default,sensor_only_smoke_no_csi_gesture,0.05,0.990000,False,True,0.520000,smoke_only,True,False,no_csi_gesture,smoke_only,True,True
1,default,csi_only_gesture_sensor_silent,0.95,0.050000,True,False,NaN,None,False,False,no_sensor_smoke,None,False,True
2,default,true_smoking_sensor_missed,0.90,0.200000,True,False,NaN,None,False,False,no_sensor_smoke,None,False,True
3,default,drink_misclassified_sensor_silent,0.80,0.200000,True,False,NaN,None,False,False,no_sensor_smoke,None,False,True
4,default,both_strong_alarm,0.95,0.990000,True,True,0.970000,fusion,True,True,alarm,fusion,True,True
5,default,both_gates_weak_csi_sensor_at_threshold,0.10,0.404999,True,True,0.252499,smoke_only,True,False,below_fused_threshold,smoke_only,True,True
6,default,sensor_compensates_moderate_csi,0.50,0.950000,True,True,0.725000,fusion,True,True,alarm,fusion,True,True
7,default,sensor_at_exact_gate,0.90,0.404999,True,True,0.652499,smoke_only,True,False,below_fused_threshold,smoke_only,True,True
8,default,score_just_above_fused_threshold,0.61,0.800000,True,True,0.705000,fusion,True,True,alarm,fusion,True,True
9,default,score_just_below_fused_threshold,0.59,0.800000,True,True,0.695000,smoke_only,True,False,below_fused_threshold,smoke_only,True,True



[pure-engine] all synthetic rule and policy checks passed.
[pure-engine] anchor score: 0.452499


## 8. Real-artifact sanity checks

Verify that the reloaded models reproduce the exported fixture probabilities, that the
CSI logit reconstruction is exact, and that calibrated probability rows sum to one.

In [ ]:
# @title 8. Real-artifact sanity checks
REAL_CHECKS = {}

if CSI_FIXTURE_AVAILABLE and SENSOR_FIXTURE_AVAILABLE:
    with np.load(CSI_FIXTURE_PATH, allow_pickle=False) as fixture:
        csi_keys = set(fixture.files)
        csi_X_check = fixture['X_test'][:8].astype(np.float32)
        csi_prob_cal_exported = (
            fixture['prob_cal'][:8].astype(np.float64)
            if 'prob_cal' in csi_keys else None
        )
        csi_smoke_cal_exported = (
            fixture['smoke_cal'][:8].astype(np.float64)
            if 'smoke_cal' in csi_keys else None
        )

    csi_check = predict_csi_proba(csi_X_check, background_subtracted=True)
    csi_model_raw = CSI_MODEL.predict(csi_X_check, verbose=0)
    csi_logits = get_model_logits(CSI_MODEL, csi_X_check)
    csi_softmax_from_logits = tf.nn.softmax(csi_logits, axis=1).numpy()

    REAL_CHECKS['csi_logit_vs_model_predict_max_abs'] = float(
        np.max(np.abs(csi_softmax_from_logits - csi_model_raw))
    )
    REAL_CHECKS['csi_calibrated_row_sum_max_abs'] = float(
        np.max(np.abs(csi_check.prob_cal.sum(axis=1) - 1.0))
    )
    REAL_CHECKS['csi_raw_vs_calibrated_argmax_disagreement'] = float(
        np.mean(csi_check.argmax_raw != csi_check.argmax_cal)
    )
    if csi_prob_cal_exported is not None:
        REAL_CHECKS['csi_prob_cal_vs_export_max_abs'] = float(
            np.max(np.abs(csi_check.prob_cal - csi_prob_cal_exported))
        )
    if csi_smoke_cal_exported is not None:
        REAL_CHECKS['csi_smoke_cal_vs_export_max_abs'] = float(
            np.max(np.abs(
                csi_check.p_smoke_binary_cal - csi_smoke_cal_exported
            ))
        )

    with np.load(SENSOR_FIXTURE_PATH, allow_pickle=False) as fixture:
        sensor_keys = set(fixture.files)
        sensor_X_check = fixture['X_test'][:8].astype(np.float64)
        sensor_prob_exported = (
            fixture['prob'][:8].astype(np.float64)
            if 'prob' in sensor_keys else None
        )
    sensor_prob_check = SENSOR_PIPELINE.predict_proba(
        sensor_X_check
    )[:, SENSOR_SMOKE_INDEX]
    if sensor_prob_exported is not None:
        REAL_CHECKS['sensor_prob_vs_export_max_abs'] = float(
            np.max(np.abs(sensor_prob_check - sensor_prob_exported))
        )
    REAL_CHECKS['sensor_prob_min'] = float(np.min(sensor_prob_check))
    REAL_CHECKS['sensor_prob_max'] = float(np.max(sensor_prob_check))

    display(pd.DataFrame([REAL_CHECKS]))
    assert REAL_CHECKS['csi_logit_vs_model_predict_max_abs'] < 1e-6
    assert REAL_CHECKS['csi_calibrated_row_sum_max_abs'] < 1e-6
    if 'csi_prob_cal_vs_export_max_abs' in REAL_CHECKS:
        assert REAL_CHECKS['csi_prob_cal_vs_export_max_abs'] < 1e-5
    if 'csi_smoke_cal_vs_export_max_abs' in REAL_CHECKS:
        assert REAL_CHECKS['csi_smoke_cal_vs_export_max_abs'] < 1e-5
    if 'sensor_prob_vs_export_max_abs' in REAL_CHECKS:
        assert REAL_CHECKS['sensor_prob_vs_export_max_abs'] < 1e-5
    print('[real-checks] model reload and fixture consistency checks passed.')
else:
    print('[real-checks] Skipped: both test fixtures are required.')

,csi_logit_vs_model_predict_max_abs,csi_calibrated_row_sum_max_abs,csi_raw_vs_calibrated_argmax_disagreement,csi_prob_cal_vs_export_max_abs,csi_smoke_cal_vs_export_max_abs,sensor_prob_vs_export_max_abs,sensor_prob_min,sensor_prob_max
0,2.980232e-08,2.220446e-16,0.0,1.379341e-07,2.384186e-07,0.0,0.000243,0.033858


[real-checks] model reload and fixture consistency checks passed.


## 9. Scenario simulations using the held-out test sets

Load the two fixtures, recompute each model's held-out probabilities, and pair CSI test
windows with sensor test intervals by **label and detector outcome**. Every pairing is
a simulation: the CSI and sensor recordings are from different sessions, so the pair
rates are not real-world co-occurrence rates or fused accuracy.

The scenario dimensions are:

- **Matrix A:** CSI true class (7) × sensor gate outcome (triggered/silent) = 14 scenarios.
- **Matrix B:** CSI true class (7) × sensor true state (smoke/no-smoke) = 14 scenarios.
- **Detector outcomes:** CSI gate × sensor gate = 4 scenarios.
- **Named risk scenarios:** the user's examples plus the known failure paths.
- **Policy sensitivity:** weight and fused-threshold variants over four anchor scenarios.

Each scenario reports the deployed **label counts** — `fusion`, `smoke_only`,
`csi_only`, `alert` (any of the three), and `no_event` — in addition to the
gate rates and score summaries.


In [ ]:
# @title 9a. Load fixtures and build scenario tables
if not (CSI_FIXTURE_AVAILABLE and SENSOR_FIXTURE_AVAILABLE):
    raise RuntimeError(
        'Both test fixtures are required and must be fresh for scenario '
        'simulations. Regenerate them with the source notebooks, then rerun '
        'from the preflight cell. See the instructions printed above.'
    )

with np.load(CSI_FIXTURE_PATH, allow_pickle=False) as fixture:
    csi_keys = set(fixture.files)
    csi_X_test = fixture['X_test'].astype(np.float32)
    csi_y_test = fixture['y_test'].astype(np.int64)
    csi_prob_cal_exported = (
        fixture['prob_cal'].astype(np.float64)
        if 'prob_cal' in csi_keys else None
    )
    csi_smoke_cal_exported = (
        fixture['smoke_cal'].astype(np.float64)
        if 'smoke_cal' in csi_keys else None
    )

csi_test_output = predict_csi_proba(csi_X_test, background_subtracted=True)
CSI_TEST = pd.DataFrame({
    'true_class': [SEVEN_CLASSES[int(value)] for value in csi_y_test],
    'pred_class': csi_test_output.argmax_cal,
    'pred_class_raw': csi_test_output.argmax_raw,
    'p_csi_smoke': csi_test_output.p_smoke_binary_cal,
    'p_csi_smoke_multiclass': csi_test_output.p_smoke_multiclass_cal,
    'csi_hit': csi_test_output.argmax_cal == 'smoke',
})

with open(CSI_FIXTURE_META_PATH, 'r', encoding='utf-8') as handle:
    CSI_FIXTURE_META = json.load(handle)
csi_segment_keys = CSI_FIXTURE_META.get('segment_keys')
if csi_segment_keys is not None and len(csi_segment_keys) == len(CSI_TEST):
    CSI_TEST['segment_key'] = csi_segment_keys

with np.load(SENSOR_FIXTURE_PATH, allow_pickle=False) as fixture:
    sensor_keys = set(fixture.files)
    sensor_X_test = fixture['X_test'].astype(np.float64)
    sensor_y_test = fixture['y_test'].astype(np.int64)
    sensor_prob_exported = (
        fixture['prob'].astype(np.float64)
        if 'prob' in sensor_keys else None
    )

sensor_prob_test = SENSOR_PIPELINE.predict_proba(
    sensor_X_test
)[:, SENSOR_SMOKE_INDEX]
SENSOR_TEST = pd.DataFrame({
    'true_state': [SENSOR_CLASSES[int(value)] for value in sensor_y_test],
    'p_sensor_smoke': sensor_prob_test,
    'sensor_hit': sensor_prob_test >= SENSOR_THRESHOLD,
})

with open(SENSOR_FIXTURE_META_PATH, 'r', encoding='utf-8') as handle:
    SENSOR_FIXTURE_META = json.load(handle)
sensor_interval_keys = SENSOR_FIXTURE_META.get('interval_keys')
if sensor_interval_keys is not None and len(sensor_interval_keys) == len(SENSOR_TEST):
    SENSOR_TEST['interval_key'] = sensor_interval_keys
sensor_session_keys = SENSOR_FIXTURE_META.get('session_keys')
if sensor_session_keys is not None and len(sensor_session_keys) == len(SENSOR_TEST):
    SENSOR_TEST['session_key'] = sensor_session_keys

SCENARIO_INPUT_CHECKS = {}
if csi_prob_cal_exported is not None:
    SCENARIO_INPUT_CHECKS['csi_prob_cal_vs_export_max_abs'] = float(
        np.max(np.abs(csi_test_output.prob_cal - csi_prob_cal_exported))
    )
if csi_smoke_cal_exported is not None:
    SCENARIO_INPUT_CHECKS['csi_smoke_cal_vs_export_max_abs'] = float(
        np.max(np.abs(
            csi_test_output.p_smoke_binary_cal - csi_smoke_cal_exported
        ))
    )
if sensor_prob_exported is not None:
    SCENARIO_INPUT_CHECKS['sensor_prob_vs_export_max_abs'] = float(
        np.max(np.abs(sensor_prob_test - sensor_prob_exported))
    )
display(pd.DataFrame([SCENARIO_INPUT_CHECKS]))
# Full-set provenance tolerance: the export and this runtime can differ by a
# few 1e-6 from float32/TF round-off. The tighter first-8 check lives in the
# real-artifact cell.
SCENARIO_INPUT_TOLERANCE = 1e-4
for key, value in SCENARIO_INPUT_CHECKS.items():
    if 'max_abs' in key:
        assert value < SCENARIO_INPUT_TOLERANCE, (
            f'{key} = {value} exceeds {SCENARIO_INPUT_TOLERANCE}'
        )

print('\nCSI held-out test set:')
display(CSI_TEST['true_class'].value_counts().rename_axis(
    'true_class').to_frame('windows'))
display(pd.crosstab(
    CSI_TEST['true_class'], CSI_TEST['pred_class'], margins=True))
print(f'CSI calibrated-gate rate: {CSI_TEST["csi_hit"].mean():.4f}')

print('\nSensor held-out test set:')
display(SENSOR_TEST['true_state'].value_counts().rename_axis(
    'true_state').to_frame('intervals'))
display(pd.crosstab(
    SENSOR_TEST['true_state'], SENSOR_TEST['sensor_hit'], margins=True))
print(f'Sensor gate rate: {SENSOR_TEST["sensor_hit"].mean():.4f}')
print(f'Shapes: CSI {csi_X_test.shape}, sensor {sensor_X_test.shape}')


,csi_prob_cal_vs_export_max_abs,csi_smoke_cal_vs_export_max_abs,sensor_prob_vs_export_max_abs
0,0.000011,0.00001,0.0



CSI held-out test set:


,windows
true_class,
static,32
smoke,32
random,32
drink,32
sit,32
stand,32
eat,32


pred_class,drink,eat,random,sit,smoke,stand,static,All
true_class,,,,,,,,
drink,20,5,3,0,4,0,0,32
eat,1,29,1,1,0,0,0,32
random,3,1,25,0,2,0,1,32
sit,0,0,1,31,0,0,0,32
smoke,3,1,3,0,25,0,0,32
stand,0,0,0,0,0,32,0,32
static,0,0,1,0,1,0,30,32
All,27,36,34,32,32,32,31,224


CSI calibrated-gate rate: 0.1429

Sensor held-out test set:


,intervals
true_state,
no-smoke,73
smoke,35


sensor_hit,False,True,All
true_state,,,
no-smoke,71,2,73
smoke,0,35,35
All,71,37,108


Sensor gate rate: 0.3426
Shapes: CSI (224, 420, 52), sensor (108, 22)


In [ ]:
# @title 9b. Scenario engine
@dataclass(frozen=True)
class ScenarioSpec:
    name: str
    description: str = ''
    csi_true_classes: tuple | None = None
    csi_pred_classes: tuple | None = None
    csi_hit: bool | None = None
    sensor_true_states: tuple | None = None
    sensor_hit: bool | None = None


@dataclass
class ScenarioResult:
    name: str
    description: str
    n_csi: int
    n_sensor: int
    n_pairs: int
    csi_hit_rate: float
    sensor_hit_rate: float
    sensor_true_smoke_share: float
    mean_score: float
    min_score: float
    max_score: float
    fusion_count: int
    smoke_only_count: int
    csi_only_count: int
    alert_count: int
    no_event_count: int
    alarm_rate: float
    reason_counts: dict
    examples: list
    simulated: bool = True

    @property
    def alarm_count(self) -> int:
        """Confirmed fusion alerts (backwards-compatible alias)."""
        return self.fusion_count

    def to_row(self) -> dict:
        reason_text = ', '.join(
            f'{key}={value}' for key, value in sorted(self.reason_counts.items())
        )
        return {
            'scenario': self.name,
            'description': self.description,
            'n_csi': self.n_csi,
            'n_sensor': self.n_sensor,
            'n_pairs': self.n_pairs,
            'csi_hit_rate': self.csi_hit_rate,
            'sensor_hit_rate': self.sensor_hit_rate,
            'sensor_true_smoke_share': self.sensor_true_smoke_share,
            'mean_score': self.mean_score,
            'min_score': self.min_score,
            'max_score': self.max_score,
            'fusion_count': self.fusion_count,
            'smoke_only_count': self.smoke_only_count,
            'csi_only_count': self.csi_only_count,
            'alert_count': self.alert_count,
            'no_event_count': self.no_event_count,
            'alarm_count': self.fusion_count,
            'alarm_rate': self.alarm_rate,
            'alert_rate': self.alert_count / float(self.n_pairs)
            if self.n_pairs else float('nan'),
            'reason_counts': reason_text,
            'simulated': self.simulated,
        }


def simulate_scenario(csi_table, sensor_table, spec,
                      policy=DEFAULT_POLICY) -> ScenarioResult:
    """Cross-product simulation: every selected CSI window x every selected sensor interval."""
    csi_selected = csi_table
    if spec.csi_true_classes is not None:
        csi_selected = csi_selected[
            csi_selected['true_class'].isin(spec.csi_true_classes)
        ]
    if spec.csi_pred_classes is not None:
        csi_selected = csi_selected[
            csi_selected['pred_class'].isin(spec.csi_pred_classes)
        ]
    if spec.csi_hit is not None:
        csi_selected = csi_selected[
            csi_selected['csi_hit'] == spec.csi_hit
        ]

    sensor_selected = sensor_table
    if spec.sensor_true_states is not None:
        sensor_selected = sensor_selected[
            sensor_selected['true_state'].isin(spec.sensor_true_states)
        ]
    if spec.sensor_hit is not None:
        sensor_selected = sensor_selected[
            sensor_selected['sensor_hit'] == spec.sensor_hit
        ]

    n_csi = int(len(csi_selected))
    n_sensor = int(len(sensor_selected))
    if n_csi == 0 or n_sensor == 0:
        return ScenarioResult(
            name=spec.name,
            description=spec.description,
            n_csi=n_csi,
            n_sensor=n_sensor,
            n_pairs=0,
            csi_hit_rate=float('nan'),
            sensor_hit_rate=float('nan'),
            sensor_true_smoke_share=float('nan'),
            mean_score=float('nan'),
            min_score=float('nan'),
            max_score=float('nan'),
            fusion_count=0,
            smoke_only_count=0,
            csi_only_count=0,
            alert_count=0,
            no_event_count=0,
            alarm_rate=float('nan'),
            reason_counts={},
            examples=[],
        )

    p_csi_pairs = np.repeat(
        csi_selected['p_csi_smoke'].to_numpy(dtype=np.float64), n_sensor
    )
    p_sensor_pairs = np.tile(
        sensor_selected['p_sensor_smoke'].to_numpy(dtype=np.float64), n_csi
    )
    if policy.csi_gate_mode == 'probability':
        csi_hit_pairs = p_csi_pairs >= policy.csi_gate_threshold
    else:
        csi_hit_pairs = np.repeat(
            csi_selected['csi_hit'].to_numpy(dtype=bool), n_sensor
        )

    fused = fuse_scores(
        p_csi_pairs, p_sensor_pairs, csi_hit_pairs, policy
    )
    fusion_mask = fused['label'].to_numpy(dtype=object) == 'fusion'
    smoke_only_mask = fused['label'].to_numpy(dtype=object) == 'smoke_only'
    csi_only_mask = fused['label'].to_numpy(dtype=object) == 'csi_only'
    alert_mask = fused['alert'].to_numpy(dtype=bool)
    no_event_mask = ~alert_mask
    fusion_count = int(fusion_mask.sum())
    smoke_only_count = int(smoke_only_mask.sum())
    csi_only_count = int(csi_only_mask.sum())
    alert_count = int(alert_mask.sum())
    no_event_count = int(no_event_mask.sum())

    score_values = fused['fused_score'].to_numpy(dtype=np.float64)
    finite_scores = score_values[np.isfinite(score_values)]
    examples = pd.DataFrame({
        'csi_true_class': np.repeat(
            csi_selected['true_class'].to_numpy(), n_sensor
        ),
        'csi_pred_class': np.repeat(
            csi_selected['pred_class'].to_numpy(), n_sensor
        ),
        'p_csi_smoke': p_csi_pairs,
        'sensor_true_state': np.tile(
            sensor_selected['true_state'].to_numpy(), n_csi
        ),
        'p_sensor_smoke': p_sensor_pairs,
        'fused_score': score_values,
        'label': fused['label'].to_numpy(),
        'alert': alert_mask,
        'reason': fused['reason'].to_numpy(),
    }).head(3).to_dict(orient='records')

    return ScenarioResult(
        name=spec.name,
        description=spec.description,
        n_csi=n_csi,
        n_sensor=n_sensor,
        n_pairs=int(len(fused)),
        csi_hit_rate=float(np.mean(csi_hit_pairs)),
        sensor_hit_rate=float(np.mean(fused['sensor_hit'])),
        sensor_true_smoke_share=float(
            np.mean(sensor_selected['true_state'] == 'smoke')
        ),
        mean_score=float(finite_scores.mean()) if finite_scores.size else float('nan'),
        min_score=float(finite_scores.min()) if finite_scores.size else float('nan'),
        max_score=float(finite_scores.max()) if finite_scores.size else float('nan'),
        fusion_count=fusion_count,
        smoke_only_count=smoke_only_count,
        csi_only_count=csi_only_count,
        alert_count=alert_count,
        no_event_count=no_event_count,
        alarm_rate=fusion_count / float(len(fused)),
        reason_counts={
            str(key): int(value)
            for key, value in fused['reason'].value_counts().items()
        },
        examples=examples,
        simulated=True,
    )


def scenario_results_to_frame(results) -> pd.DataFrame:
    return pd.DataFrame([result.to_row() for result in results])


print('[scenarios] engine ready (backend labels: fusion / smoke_only / csi_only).')


[scenarios] engine ready (backend labels: fusion / smoke_only / csi_only).


In [ ]:
# @title 9c. Scenario matrices (14 + 14 + 4)
MATRIX_A_SPECS = []
for class_name in SEVEN_CLASSES:
    for sensor_hit in (True, False):
        suffix = 'sensor_triggered' if sensor_hit else 'sensor_silent'
        MATRIX_A_SPECS.append(ScenarioSpec(
            name=f'{class_name}_csi__{suffix}',
            description='CSI true class x sensor gate outcome',
            csi_true_classes=(class_name,),
            sensor_hit=sensor_hit,
        ))
MATRIX_A_RESULTS = [
    simulate_scenario(CSI_TEST, SENSOR_TEST, spec)
    for spec in MATRIX_A_SPECS
]
MATRIX_A_DF = scenario_results_to_frame(MATRIX_A_RESULTS)
print('Matrix A — CSI true class x sensor gate outcome (simulated):')
display(MATRIX_A_DF.round(4))

MATRIX_B_SPECS = []
for class_name in SEVEN_CLASSES:
    for sensor_state in SENSOR_CLASSES:
        MATRIX_B_SPECS.append(ScenarioSpec(
            name=f'{class_name}_csi__sensor_true_{sensor_state}',
            description='CSI true class x sensor true state',
            csi_true_classes=(class_name,),
            sensor_true_states=(sensor_state,),
        ))
MATRIX_B_RESULTS = [
    simulate_scenario(CSI_TEST, SENSOR_TEST, spec)
    for spec in MATRIX_B_SPECS
]
MATRIX_B_DF = scenario_results_to_frame(MATRIX_B_RESULTS)
print('\nMatrix B — CSI true class x sensor true state (simulated):')
display(MATRIX_B_DF.round(4))

DETECTOR_SPECS = []
for csi_hit in (True, False):
    for sensor_hit in (True, False):
        DETECTOR_SPECS.append(ScenarioSpec(
            name=f'csi_gate_{"hit" if csi_hit else "miss"}__sensor_gate_{"hit" if sensor_hit else "miss"}',
            description='CSI gate outcome x sensor gate outcome',
            csi_hit=csi_hit,
            sensor_hit=sensor_hit,
        ))
DETECTOR_RESULTS = [
    simulate_scenario(CSI_TEST, SENSOR_TEST, spec)
    for spec in DETECTOR_SPECS
]
DETECTOR_DF = scenario_results_to_frame(DETECTOR_RESULTS)
print('\nDetector-outcome matrix (simulated):')
display(DETECTOR_DF.round(4))

print('\nStructural checks:')
for result in DETECTOR_RESULTS:
    if not result.sensor_hit_rate:
        assert result.fusion_count == 0, (
            f'{result.name} produced fusion alarms with the sensor gate off.'
        )
        assert result.smoke_only_count == 0, (
            f'{result.name} produced smoke_only alerts with the sensor gate off.'
        )
        if not DEFAULT_POLICY.csi_only_enabled:
            assert result.csi_only_count == 0, (
                f'{result.name} produced csi_only alerts while disabled.'
            )
    if not result.csi_hit_rate:
        assert result.fusion_count == 0, (
            f'{result.name} produced fusion alarms with the CSI gate off.'
        )
        assert result.csi_only_count == 0, (
            f'{result.name} produced csi_only alerts with the CSI gate off.'
        )
print('  sensor-gate-off and CSI-gate-off scenarios have no impossible labels.')


Matrix A — CSI true class x sensor gate outcome (simulated):


,scenario,description,n_csi,n_sensor,n_pairs,csi_hit_rate,sensor_hit_rate,sensor_true_smoke_share,mean_score,min_score,max_score,fusion_count,smoke_only_count,csi_only_count,alert_count,no_event_count,alarm_count,alarm_rate,alert_rate,reason_counts,simulated
0,smoke_csi__sensor_triggered,CSI true class x sensor gate outcome,32,37,1184,0.7812,1.0,0.9459,0.8040,0.2153,0.9738,840,344,0,1184,0,840,0.7095,1.0,"alarm=840, below_fused_threshold=85, no_csi_gesture=259",True
1,smoke_csi__sensor_silent,CSI true class x sensor gate outcome,32,71,2272,0.7812,0.0,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0,no_sensor_smoke=2272,True
2,eat_csi__sensor_triggered,CSI true class x sensor gate outcome,32,37,1184,0.0000,1.0,0.9459,0.4696,0.2149,0.5318,0,1184,0,1184,0,0,0.0000,1.0,no_csi_gesture=1184,True
3,eat_csi__sensor_silent,CSI true class x sensor gate outcome,32,71,2272,0.0000,0.0,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0,no_sensor_smoke=2272,True
4,drink_csi__sensor_triggered,CSI true class x sensor gate outcome,32,37,1184,0.1250,1.0,0.9459,0.5199,0.2149,0.9517,105,1079,0,1184,0,105,0.0887,1.0,"alarm=105, below_fused_threshold=43, no_csi_gesture=1036",True
5,drink_csi__sensor_silent,CSI true class x sensor gate outcome,32,71,2272,0.1250,0.0,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0,no_sensor_smoke=2272,True
6,sit_csi__sensor_triggered,CSI true class x sensor gate outcome,32,37,1184,0.0000,1.0,0.9459,0.4729,0.2160,0.5260,0,1184,0,1184,0,0,0.0000,1.0,no_csi_gesture=1184,True
7,sit_csi__sensor_silent,CSI true class x sensor gate outcome,32,71,2272,0.0000,0.0,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0,no_sensor_smoke=2272,True
8,stand_csi__sensor_triggered,CSI true class x sensor gate outcome,32,37,1184,0.0000,1.0,0.9459,0.4705,0.2159,0.5179,0,1184,0,1184,0,0,0.0000,1.0,no_csi_gesture=1184,True
9,stand_csi__sensor_silent,CSI true class x sensor gate outcome,32,71,2272,0.0000,0.0,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0,no_sensor_smoke=2272,True



Matrix B — CSI true class x sensor true state (simulated):


,scenario,description,n_csi,n_sensor,n_pairs,csi_hit_rate,sensor_hit_rate,sensor_true_smoke_share,mean_score,min_score,max_score,fusion_count,smoke_only_count,csi_only_count,alert_count,no_event_count,alarm_count,alarm_rate,alert_rate,reason_counts,simulated
0,smoke_csi__sensor_true_no-smoke,CSI true class x sensor true state,32,73,2336,0.7812,0.0274,0.0,0.5543,0.2153,0.6924,0,64,0,64,2272,0,0.0000,0.0274,"below_fused_threshold=50, no_csi_gesture=14, no_sensor_smoke=2272",True
1,smoke_csi__sensor_true_smoke,CSI true class x sensor true state,32,35,1120,0.7812,1.0000,1.0,0.8183,0.4262,0.9738,840,280,0,1120,0,840,0.7500,1.0000,"alarm=840, below_fused_threshold=35, no_csi_gesture=245",True
2,eat_csi__sensor_true_no-smoke,CSI true class x sensor true state,32,73,2336,0.0000,0.0274,0.0,0.2199,0.2149,0.2503,0,64,0,64,2272,0,0.0000,0.0274,"no_csi_gesture=64, no_sensor_smoke=2272",True
3,eat_csi__sensor_true_smoke,CSI true class x sensor true state,32,35,1120,0.0000,1.0000,1.0,0.4839,0.4258,0.5318,0,1120,0,1120,0,0,0.0000,1.0000,no_csi_gesture=1120,True
4,drink_csi__sensor_true_no-smoke,CSI true class x sensor true state,32,73,2336,0.1250,0.0274,0.0,0.2702,0.2149,0.6702,0,64,0,64,2272,0,0.0000,0.0274,"below_fused_threshold=8, no_csi_gesture=56, no_sensor_smoke=2272",True
5,drink_csi__sensor_true_smoke,CSI true class x sensor true state,32,35,1120,0.1250,1.0000,1.0,0.5341,0.4257,0.9517,105,1015,0,1120,0,105,0.0938,1.0000,"alarm=105, below_fused_threshold=35, no_csi_gesture=980",True
6,sit_csi__sensor_true_no-smoke,CSI true class x sensor true state,32,73,2336,0.0000,0.0274,0.0,0.2232,0.2160,0.2445,0,64,0,64,2272,0,0.0000,0.0274,"no_csi_gesture=64, no_sensor_smoke=2272",True
7,sit_csi__sensor_true_smoke,CSI true class x sensor true state,32,35,1120,0.0000,1.0000,1.0,0.4871,0.4269,0.5260,0,1120,0,1120,0,0,0.0000,1.0000,no_csi_gesture=1120,True
8,stand_csi__sensor_true_no-smoke,CSI true class x sensor true state,32,73,2336,0.0000,0.0274,0.0,0.2208,0.2159,0.2364,0,64,0,64,2272,0,0.0000,0.0274,"no_csi_gesture=64, no_sensor_smoke=2272",True
9,stand_csi__sensor_true_smoke,CSI true class x sensor true state,32,35,1120,0.0000,1.0000,1.0,0.4848,0.4268,0.5179,0,1120,0,1120,0,0,0.0000,1.0000,no_csi_gesture=1120,True



Detector-outcome matrix (simulated):


,scenario,description,n_csi,n_sensor,n_pairs,csi_hit_rate,sensor_hit_rate,sensor_true_smoke_share,mean_score,min_score,max_score,fusion_count,smoke_only_count,csi_only_count,alert_count,no_event_count,alarm_count,alarm_rate,alert_rate,reason_counts,simulated
0,csi_gate_hit__sensor_gate_hit,CSI gate outcome x sensor gate outcome,32,37,1184,1.0,1.0,0.9459,0.8697,0.3523,0.9738,1015,169,0,1184,0,1015,0.8573,1.0,"alarm=1015, below_fused_threshold=169",True
1,csi_gate_hit__sensor_gate_miss,CSI gate outcome x sensor gate outcome,32,71,2272,1.0,0.0,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0,no_sensor_smoke=2272,True
2,csi_gate_miss__sensor_gate_hit,CSI gate outcome x sensor gate outcome,192,37,7104,0.0,1.0,0.9459,0.4746,0.2149,0.6612,0,7104,0,7104,0,0,0.0000,1.0,no_csi_gesture=7104,True
3,csi_gate_miss__sensor_gate_miss,CSI gate outcome x sensor gate outcome,192,71,13632,0.0,0.0,0.0000,NaN,NaN,NaN,0,0,0,0,13632,0,0.0000,0.0,no_sensor_smoke=13632,True



Structural checks:
  sensor-gate-off and CSI-gate-off scenarios have no impossible labels.


In [ ]:
# @title 9d. Named risk scenarios
NAMED_SPECS = [
    ScenarioSpec(
        'smoking_activity_sensor_triggered',
        'True smoking windows paired with sensor-triggered intervals.',
        csi_true_classes=('smoke',), sensor_hit=True),
    ScenarioSpec(
        'smoking_activity_sensor_missed',
        'True smoking windows paired with sensor-silent intervals; the accepted compromise.',
        csi_true_classes=('smoke',), sensor_hit=False),
    ScenarioSpec(
        'smoking_activity_sensor_false_negative',
        'True smoking windows paired with sensor true no-smoke intervals.',
        csi_true_classes=('smoke',), sensor_true_states=('no-smoke',)),
    ScenarioSpec(
        'drinking_activity_sensor_triggered',
        'True drinking windows paired with sensor-triggered intervals.',
        csi_true_classes=('drink',), sensor_hit=True),
    ScenarioSpec(
        'drinking_activity_sensor_silent',
        'True drinking windows paired with sensor-silent intervals.',
        csi_true_classes=('drink',), sensor_hit=False),
    ScenarioSpec(
        'drinking_misclassified_as_smoke_sensor_triggered',
        'The key false-positive path: drinking windows the CSI model calls smoke, with the sensor triggered.',
        csi_true_classes=('drink',), csi_hit=True, sensor_hit=True),
    ScenarioSpec(
        'sitting_activity_sensor_silent',
        'True sitting windows paired with sensor-silent intervals.',
        csi_true_classes=('sit',), sensor_hit=False),
    ScenarioSpec(
        'sitting_activity_sensor_triggered',
        'True sitting windows paired with sensor-triggered intervals.',
        csi_true_classes=('sit',), sensor_hit=True),
    ScenarioSpec(
        'eating_activity_sensor_triggered',
        'True eating windows paired with sensor-triggered intervals.',
        csi_true_classes=('eat',), sensor_hit=True),
    ScenarioSpec(
        'standing_activity_sensor_triggered',
        'True standing windows paired with sensor-triggered intervals.',
        csi_true_classes=('stand',), sensor_hit=True),
    ScenarioSpec(
        'random_activity_sensor_triggered',
        'True random/hand-wave windows paired with sensor-triggered intervals.',
        csi_true_classes=('random',), sensor_hit=True),
    ScenarioSpec(
        'sensor_false_alarm_no_csi_gesture',
        'Sensor false alarms (true no-smoke but sensor triggered) with no CSI gesture.',
        csi_hit=False, sensor_true_states=('no-smoke',), sensor_hit=True),
    ScenarioSpec(
        'no_activity_no_sensor',
        'Sitting/standing/random/static windows paired with sensor true no-smoke intervals.',
        csi_true_classes=('sit', 'stand', 'random', 'static'),
        sensor_true_states=('no-smoke',)),
]
NAMED_RESULTS = [
    simulate_scenario(CSI_TEST, SENSOR_TEST, spec)
    for spec in NAMED_SPECS
]
NAMED_DF = scenario_results_to_frame(NAMED_RESULTS)
print('Named risk scenarios (simulated):')
display(NAMED_DF.round(4))

KEY_EXAMPLE_SCENARIOS = {
    'smoking_activity_sensor_triggered',
    'smoking_activity_sensor_missed',
    'drinking_activity_sensor_triggered',
    'drinking_misclassified_as_smoke_sensor_triggered',
    'sitting_activity_sensor_silent',
    'sensor_false_alarm_no_csi_gesture',
}
for result in NAMED_RESULTS:
    if result.name in KEY_EXAMPLE_SCENARIOS and result.examples:
        print(f'\nExamples — {result.name} '
              f'(alarm_rate={result.alarm_rate:.4f}, n_pairs={result.n_pairs})')
        display(pd.DataFrame(result.examples).round(4))

# Structural expectations for the deployed labels.
NAMED_BY_NAME = {result.name: result for result in NAMED_RESULTS}
for name in ('smoking_activity_sensor_missed',
             'drinking_activity_sensor_silent',
             'sitting_activity_sensor_silent'):
    assert NAMED_BY_NAME[name].alert_count == 0, (
        f'{name} should never alert with the sensor gate off '
        '(csi_only is disabled by default).'
    )
assert NAMED_BY_NAME['smoking_activity_sensor_false_negative'].csi_only_count == 0, (
    'csi_only must be disabled by default.'
)
assert NAMED_BY_NAME['sensor_false_alarm_no_csi_gesture'].fusion_count == 0, (
    'fusion cannot fire without the CSI gesture gate.'
)
assert NAMED_BY_NAME['sensor_false_alarm_no_csi_gesture'].csi_only_count == 0, (
    'csi_only cannot fire without the CSI gesture gate.'
)
print('\n[named] deployed-label structural expectations hold.')


Named risk scenarios (simulated):


,scenario,description,n_csi,n_sensor,n_pairs,csi_hit_rate,sensor_hit_rate,sensor_true_smoke_share,mean_score,min_score,max_score,fusion_count,smoke_only_count,csi_only_count,alert_count,no_event_count,alarm_count,alarm_rate,alert_rate,reason_counts,simulated
0,smoking_activity_sensor_triggered,True smoking windows paired with sensor-triggered intervals.,32,37,1184,0.7812,1.0000,0.9459,0.8040,0.2153,0.9738,840,344,0,1184,0,840,0.7095,1.0000,"alarm=840, below_fused_threshold=85, no_csi_gesture=259",True
1,smoking_activity_sensor_missed,True smoking windows paired with sensor-silent intervals; the accepted compromise.,32,71,2272,0.7812,0.0000,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0000,no_sensor_smoke=2272,True
2,smoking_activity_sensor_false_negative,True smoking windows paired with sensor true no-smoke intervals.,32,73,2336,0.7812,0.0274,0.0000,0.5543,0.2153,0.6924,0,64,0,64,2272,0,0.0000,0.0274,"below_fused_threshold=50, no_csi_gesture=14, no_sensor_smoke=2272",True
3,drinking_activity_sensor_triggered,True drinking windows paired with sensor-triggered intervals.,32,37,1184,0.1250,1.0000,0.9459,0.5199,0.2149,0.9517,105,1079,0,1184,0,105,0.0887,1.0000,"alarm=105, below_fused_threshold=43, no_csi_gesture=1036",True
4,drinking_activity_sensor_silent,True drinking windows paired with sensor-silent intervals.,32,71,2272,0.1250,0.0000,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0000,no_sensor_smoke=2272,True
5,drinking_misclassified_as_smoke_sensor_triggered,"The key false-positive path: drinking windows the CSI model calls smoke, with the sensor triggered.",4,37,148,1.0000,1.0000,0.9459,0.8354,0.3523,0.9517,105,43,0,148,0,105,0.7095,1.0000,"alarm=105, below_fused_threshold=43",True
6,sitting_activity_sensor_silent,True sitting windows paired with sensor-silent intervals.,32,71,2272,0.0000,0.0000,0.0000,NaN,NaN,NaN,0,0,0,0,2272,0,0.0000,0.0000,no_sensor_smoke=2272,True
7,sitting_activity_sensor_triggered,True sitting windows paired with sensor-triggered intervals.,32,37,1184,0.0000,1.0000,0.9459,0.4729,0.2160,0.5260,0,1184,0,1184,0,0,0.0000,1.0000,no_csi_gesture=1184,True
8,eating_activity_sensor_triggered,True eating windows paired with sensor-triggered intervals.,32,37,1184,0.0000,1.0000,0.9459,0.4696,0.2149,0.5318,0,1184,0,1184,0,0,0.0000,1.0000,no_csi_gesture=1184,True
9,standing_activity_sensor_triggered,True standing windows paired with sensor-triggered intervals.,32,37,1184,0.0000,1.0000,0.9459,0.4705,0.2159,0.5179,0,1184,0,1184,0,0,0.0000,1.0000,no_csi_gesture=1184,True



Examples — smoking_activity_sensor_triggered (alarm_rate=0.7095, n_pairs=1184)


,csi_true_class,csi_pred_class,p_csi_smoke,sensor_true_state,p_sensor_smoke,fused_score,label,alert,reason
0,smoke,smoke,0.7446,smoke,0.9346,0.8396,fusion,True,alarm
1,smoke,smoke,0.7446,smoke,0.9904,0.8675,fusion,True,alarm
2,smoke,smoke,0.7446,smoke,0.9501,0.8473,fusion,True,alarm



Examples — smoking_activity_sensor_missed (alarm_rate=0.0000, n_pairs=2272)


,csi_true_class,csi_pred_class,p_csi_smoke,sensor_true_state,p_sensor_smoke,fused_score,label,alert,reason
0,smoke,smoke,0.7446,no-smoke,0.0004,NaN,None,False,no_sensor_smoke
1,smoke,smoke,0.7446,no-smoke,0.0002,NaN,None,False,no_sensor_smoke
2,smoke,smoke,0.7446,no-smoke,0.0004,NaN,None,False,no_sensor_smoke


... (5 outputs left out)

,csi_true_class,csi_pred_class,p_csi_smoke,sensor_true_state,p_sensor_smoke,fused_score,label,alert,reason
0,sit,sit,0.0138,no-smoke,0.0004,NaN,None,False,no_sensor_smoke
1,sit,sit,0.0138,no-smoke,0.0002,NaN,None,False,no_sensor_smoke
2,sit,sit,0.0138,no-smoke,0.0004,NaN,None,False,no_sensor_smoke



Examples — sensor_false_alarm_no_csi_gesture (alarm_rate=0.0000, n_pairs=384)


,csi_true_class,csi_pred_class,p_csi_smoke,sensor_true_state,p_sensor_smoke,fused_score,label,alert,reason
0,static,static,0.0070,no-smoke,0.4342,0.2206,smoke_only,True,no_csi_gesture
1,static,static,0.0070,no-smoke,0.4292,0.2181,smoke_only,True,no_csi_gesture
2,random,random,0.0337,no-smoke,0.4342,0.2339,smoke_only,True,no_csi_gesture



[named] deployed-label structural expectations hold.


In [ ]:
# @title 9e. Policy sensitivity (weights x fused threshold)
ANCHOR_SPECS = [
    ScenarioSpec(
        'smoking_activity_sensor_triggered',
        csi_true_classes=('smoke',), sensor_hit=True),
    ScenarioSpec(
        'drinking_activity_sensor_triggered',
        csi_true_classes=('drink',), sensor_hit=True),
    ScenarioSpec(
        'smoking_activity_sensor_missed',
        csi_true_classes=('smoke',), sensor_hit=False),
    ScenarioSpec(
        'sitting_activity_sensor_silent',
        csi_true_classes=('sit',), sensor_hit=False),
]

SENSITIVITY_ROWS = []
for weight_csi, weight_sensor in ((0.4, 0.6), (0.3, 0.7), (0.2, 0.8)):
    for fused_threshold in (0.60, 0.65, 0.70, 0.75):
        variant = replace(
            DEFAULT_POLICY,
            weight_csi=weight_csi,
            weight_sensor=weight_sensor,
            fused_threshold=fused_threshold,
        )
        for spec in ANCHOR_SPECS:
            result = simulate_scenario(
                CSI_TEST, SENSOR_TEST, spec, policy=variant
            )
            SENSITIVITY_ROWS.append({
                'weight_csi': weight_csi,
                'weight_sensor': weight_sensor,
                'fused_threshold': fused_threshold,
                'scenario': spec.name,
                'n_pairs': result.n_pairs,
                'alarm_count': result.alarm_count,
                'alarm_rate': result.alarm_rate,
            })
SENSITIVITY_DF = pd.DataFrame(SENSITIVITY_ROWS)
SENSITIVITY_PIVOT = SENSITIVITY_DF.pivot_table(
    index=['weight_csi', 'weight_sensor', 'fused_threshold'],
    columns='scenario',
    values='alarm_rate',
)
print('Policy sensitivity — alarm rates on anchor scenarios (simulated):')
display(SENSITIVITY_PIVOT.round(4))
print(
    'Reading: raising the sensor weight/fused threshold trades sensitivity '
    'on true smoking for fewer drinking-triggered alarms. These are simulated '
    'pairings, not tuned real-world rates.'
)

Policy sensitivity — alarm rates on anchor scenarios (simulated):


scenario                                  drinking_activity_sensor_triggered  sitting_activity_sensor_silent  smoking_activity_sensor_missed  smoking_activity_sensor_triggered
weight_csi weight_sensor fused_threshold                                                                                                                                       
0.2        0.8           0.60                                         0.1182                             0.0                             0.0                             0.7390
                         0.65                                         0.1182                             0.0                             0.0                             0.7390
                         0.70                                         0.1182                             0.0                             0.0                             0.7390
                         0.75                                         0.1174                             0.0                             0.0                             0.7382
0.3        0.7           0.60                                         0.1182                             0.0                             0.0                             0.7390
                         0.65                                         0.1182                             0.0                             0.0                             0.7390
                         0.70                                         0.1174                             0.0                             0.0                             0.7382
                         0.75                                         0.1081                             0.0                             0.0                             0.7331
0.4        0.6           0.60                                         0.1233                             0.0                             0.0                             0.7677
                         0.65                                         0.1174                             0.0                             0.0                             0.7382
                         0.70                                         0.0954                             0.0                             0.0                             0.7264
                         0.75                                         0.0887                             0.0                             0.0                             0.7095

Reading: raising the sensor weight/fused threshold trades sensitivity on true smoking for fewer drinking-triggered alarms. These are simulated pairings, not tuned real-world rates.


In [ ]:
# @title 9f. Persist scenario results
SCENARIO_FRAMES = {
    'matrix_a_csi_true_x_sensor_gate': MATRIX_A_DF,
    'matrix_b_csi_true_x_sensor_true': MATRIX_B_DF,
    'detector_outcome': DETECTOR_DF,
    'named_scenarios': NAMED_DF,
}
for table_name, frame in SCENARIO_FRAMES.items():
    frame.insert(0, 'table', table_name)

SCENARIO_ALL = pd.concat(
    list(SCENARIO_FRAMES.values()), ignore_index=True
)
SCENARIO_ALL['simulated'] = True
SCENARIO_ALL['paired_data_available'] = False

SCENARIO_JSON = {
    'schema_version': FUSION_SCHEMA_VERSION,
    'created_at_utc': utc_now_iso(),
    'simulated': True,
    'paired_data_available': False,
    'fused_metrics': 'not_computed',
    'caveat': (
        'CSI and sensor test sets are from different sessions and were paired '
        'by label/outcome. These are simulations, not fused accuracy or '
        'real-world co-occurrence rates.'
    ),
    'policy': DEFAULT_POLICY.to_dict(),
    'fixture_paths': {
        'csi': CSI_FIXTURE_PATH,
        'sensor': SENSOR_FIXTURE_PATH,
    },
    'scenarios': frame_records(SCENARIO_ALL),
    'sensitivity': frame_records(SENSITIVITY_DF),
}
os.makedirs(FUSION_DIR, exist_ok=True)
atomic_write_text(
    FUSION_SCENARIOS_CSV_PATH, SCENARIO_ALL.to_csv(index=False)
)
atomic_write_json(FUSION_SCENARIOS_JSON_PATH, SCENARIO_JSON)
SCENARIO_ALL_RELOADED = pd.read_csv(FUSION_SCENARIOS_CSV_PATH)
assert len(SCENARIO_ALL_RELOADED) == len(SCENARIO_ALL)
with open(FUSION_SCENARIOS_JSON_PATH, 'r', encoding='utf-8') as handle:
    assert json.load(handle)['simulated'] is True
print(f'[scenarios] saved {len(SCENARIO_ALL)} scenario rows to '
      f'{FUSION_SCENARIOS_CSV_PATH}')
print(f'[scenarios] saved metadata to {FUSION_SCENARIOS_JSON_PATH}')


[scenarios] saved 45 scenario rows to /content/drive/MyDrive/CSI_TRAINING/models/fusion_v1/fusion_scenarios.csv
[scenarios] saved metadata to /content/drive/MyDrive/CSI_TRAINING/models/fusion_v1/fusion_scenarios.json


## 10. Decision-surface audit

A table-only audit of the deployed alarm region. It shows the two-dimensional
gate structure: `fusion` only where the CSI gate is on, the sensor gate is on,
and the weighted score clears `0.70`; `smoke_only` wherever the sensor gate is
on without a fusion confirmation; `csi_only` only when the experimental toggle
is enabled and the sensor is below its gate. This is a policy audit, not a
model evaluation.


In [ ]:
# @title 10. Decision-surface audit
AUDIT_ROWS = []
for csi_hit in (False, True):
    for p_sensor in (0.0, 0.25, 0.50, 0.75,
                     float(SENSOR_THRESHOLD), 0.80, 0.90, 1.0):
        p_csi = 0.90 if csi_hit else 0.05
        outcome = decide(p_csi, p_sensor, csi_hit, DEFAULT_POLICY)
        AUDIT_ROWS.append({
            'csi_hit': csi_hit,
            'p_csi_smoke': p_csi,
            'p_sensor_smoke': p_sensor,
            'sensor_hit': outcome.sensor_hit,
            'fused_score': outcome.fused_score,
            'label': outcome.label,
            'alert': outcome.alert,
            'alarm': outcome.alarm,
            'reason': outcome.reason,
        })
AUDIT_DF = pd.DataFrame(AUDIT_ROWS)
print('Policy audit — argmax CSI gate x sensor probability (deployed labels):')
display(AUDIT_DF.round(4))

PROB_POLICY = replace(
    DEFAULT_POLICY,
    csi_gate_mode='probability',
    csi_gate_threshold=0.50,
    allow_sub_anchor=True,
)
PROB_GRID = np.round(np.arange(0.0, 1.0001, 0.1), 2)
PROB_ROWS = []
for p_csi in PROB_GRID:
    for p_sensor in PROB_GRID:
        csi_hit = bool(p_csi >= PROB_POLICY.csi_gate_threshold)
        outcome = decide(
            float(p_csi), float(p_sensor), csi_hit, PROB_POLICY
        )
        PROB_ROWS.append({
            'p_csi_smoke': float(p_csi),
            'p_sensor_smoke': float(p_sensor),
            'label': outcome.label,
            'alert': int(outcome.alert),
        })
PROB_GRID_DF = pd.DataFrame(PROB_ROWS)
PROB_PIVOT = PROB_GRID_DF.pivot(
    index='p_csi_smoke', columns='p_sensor_smoke', values='alert'
)
print('\nPolicy audit — probability-mode alert region (1 = alert):')
display(PROB_PIVOT.astype(int))


Policy audit — argmax CSI gate x sensor probability (deployed labels):


,csi_hit,p_csi_smoke,p_sensor_smoke,sensor_hit,fused_score,label,alert,alarm,reason
0,False,0.05,0.000,False,NaN,None,False,False,no_sensor_smoke
1,False,0.05,0.250,False,NaN,None,False,False,no_sensor_smoke
2,False,0.05,0.500,True,0.2750,smoke_only,True,False,no_csi_gesture
3,False,0.05,0.750,True,0.4000,smoke_only,True,False,no_csi_gesture
4,False,0.05,0.405,True,0.2275,smoke_only,True,False,no_csi_gesture
5,False,0.05,0.800,True,0.4250,smoke_only,True,False,no_csi_gesture
6,False,0.05,0.900,True,0.4750,smoke_only,True,False,no_csi_gesture
7,False,0.05,1.000,True,0.5250,smoke_only,True,False,no_csi_gesture
8,True,0.90,0.000,False,NaN,None,False,False,no_sensor_smoke
9,True,0.90,0.250,False,NaN,None,False,False,no_sensor_smoke



Policy audit — probability-mode alert region (1 = alert):


p_sensor_smoke,0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1.0
p_csi_smoke,,,,,,,,,,,
0.0,0,0,0,0,0,1,1,1,1,1,1
0.1,0,0,0,0,0,1,1,1,1,1,1
0.2,0,0,0,0,0,1,1,1,1,1,1
0.3,0,0,0,0,0,1,1,1,1,1,1
0.4,0,0,0,0,0,1,1,1,1,1,1
0.5,0,0,0,0,0,1,1,1,1,1,1
0.6,0,0,0,0,0,1,1,1,1,1,1
0.7,0,0,0,0,0,1,1,1,1,1,1
0.8,0,0,0,0,0,1,1,1,1,1,1


## 11. Paired-data evaluation harness

This section is deliberately **not run on the simulated pairs**. When synchronized
CSI+sensor recordings exist, build a table with:

| Column | Meaning |
|---|---|
| `timestamp_utc` | decision time shared by both modalities |
| `p_csi_smoke` | calibrated binary `P(smoke)` from the CSI model |
| `p_sensor_smoke` | calibrated `P(smoke)` from the sensor SVM |
| `csi_hit` or `csi_pred_class` | CSI gesture gate |
| `y_true` | 1 if a smoking activity occurred in the environment during the window |

Then call `evaluate_fusion(table, policy)` and `sweep_fused_threshold(...)`. This is
the only way to obtain honest fused accuracy/ROC and to tune weights/thresholds.

In [ ]:
# @title 11. Paired-data evaluation harness
PAIRED_REQUIRED_COLUMNS = {'p_csi_smoke', 'p_sensor_smoke', 'y_true'}


def _paired_csi_hit(table):
    if 'csi_hit' in table.columns:
        return table['csi_hit'].astype(bool).to_numpy()
    if 'csi_pred_class' in table.columns:
        return (table['csi_pred_class'].astype(str) == 'smoke').to_numpy()
    raise ValueError(
        'Paired table needs csi_hit or csi_pred_class to define the CSI gate.'
    )


def evaluate_fusion(table, policy=DEFAULT_POLICY) -> dict:
    """Metrics for a genuinely paired CSI+sensor decision table.

    The primary detection event is the deployed alert (fusion, smoke_only, or
    csi_only); fusion-only counts are reported separately.
    """
    missing = PAIRED_REQUIRED_COLUMNS - set(table.columns)
    if missing:
        raise ValueError(f'Paired table is missing columns: {sorted(missing)}')
    if len(table) == 0:
        raise ValueError('Paired table is empty.')
    y_true = table['y_true'].to_numpy(dtype=int)
    if not set(np.unique(y_true)).issubset({0, 1}):
        raise ValueError('y_true must be binary (0/1).')
    fused = fuse_scores(
        table['p_csi_smoke'].to_numpy(dtype=np.float64),
        table['p_sensor_smoke'].to_numpy(dtype=np.float64),
        _paired_csi_hit(table),
        policy,
    )
    alert = fused['alert'].to_numpy(dtype=bool)
    tp = int(np.sum(alert & (y_true == 1)))
    fp = int(np.sum(alert & (y_true == 0)))
    fn = int(np.sum(~alert & (y_true == 1)))
    tn = int(np.sum(~alert & (y_true == 0)))
    precision = tp / (tp + fp) if (tp + fp) else float('nan')
    recall = tp / (tp + fn) if (tp + fn) else float('nan')
    f1 = (2.0 * precision * recall / (precision + recall)
          if (precision + recall) and np.isfinite(precision)
          and np.isfinite(recall) else float('nan'))
    labels = fused['label'].to_numpy(dtype=object)
    return {
        'n': int(len(table)),
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'false_alarm_rate': fp / (fp + tn) if (fp + tn) else float('nan'),
        'missed_detection_rate': fn / (fn + tp) if (fn + tp) else float('nan'),
        'accuracy': (tp + tn) / len(table),
        'fusion_count': int(np.sum(labels == 'fusion')),
        'smoke_only_count': int(np.sum(labels == 'smoke_only')),
        'csi_only_count': int(np.sum(labels == 'csi_only')),
        'alert_count': int(alert.sum()),
        'reason_counts': {
            str(key): int(value)
            for key, value in fused['reason'].value_counts().items()
        },
    }


def sweep_fused_threshold(table, policy=DEFAULT_POLICY,
                          thresholds=(0.50, 0.55, 0.60, 0.65,
                                      0.70, 0.75, 0.80)) -> pd.DataFrame:
    """Threshold sweep for a genuinely paired table."""
    rows = []
    for threshold in thresholds:
        variant = replace(policy, fused_threshold=float(threshold))
        metrics = evaluate_fusion(table, variant)
        rows.append({
            'fused_threshold': float(threshold),
            'precision': metrics['precision'],
            'recall': metrics['recall'],
            'f1': metrics['f1'],
            'false_alarm_rate': metrics['false_alarm_rate'],
            'missed_detection_rate': metrics['missed_detection_rate'],
        })
    return pd.DataFrame(rows)


PAIRED_SELF_TEST = pd.DataFrame({
    'p_csi_smoke': [0.90, 0.90, 0.10, 0.10, 0.90, 0.10],
    'p_sensor_smoke': [0.90, 0.10, 0.90, 0.10, 0.20, 0.90],
    'csi_hit': [True, True, False, False, True, False],
    'y_true': [1, 1, 0, 0, 1, 0],
})
PAIRED_SELF_METRICS = evaluate_fusion(PAIRED_SELF_TEST, DEFAULT_POLICY)
assert PAIRED_SELF_METRICS['tp'] == 1
assert PAIRED_SELF_METRICS['fp'] == 2
assert PAIRED_SELF_METRICS['fn'] == 2
assert PAIRED_SELF_METRICS['tn'] == 1
assert PAIRED_SELF_METRICS['fusion_count'] == 1
assert PAIRED_SELF_METRICS['smoke_only_count'] == 2
assert PAIRED_SELF_METRICS['csi_only_count'] == 0
assert abs(PAIRED_SELF_METRICS['precision'] - 1.0 / 3.0) < 1e-12
assert abs(PAIRED_SELF_METRICS['recall'] - 1.0 / 3.0) < 1e-12
print('[paired-harness] self-test passed (deployed alert = fusion/smoke_only/csi_only):')
display(pd.DataFrame([PAIRED_SELF_METRICS]))
print('[paired-harness] threshold sweep self-test:')
display(sweep_fused_threshold(PAIRED_SELF_TEST, DEFAULT_POLICY).round(4))
print('[paired-harness] Not run on simulated pairs — no fused metric is claimed.')


[paired-harness] self-test passed (deployed alert = fusion/smoke_only/csi_only):


,n,tp,fp,fn,tn,precision,recall,f1,false_alarm_rate,missed_detection_rate,accuracy,fusion_count,smoke_only_count,csi_only_count,alert_count,reason_counts
0,6,1,2,2,1,0.333333,0.333333,0.333333,0.666667,0.666667,0.333333,1,2,0,3,"{'no_sensor_smoke': 3, 'no_csi_gesture': 2, 'alarm': 1}"


[paired-harness] threshold sweep self-test:


,fused_threshold,precision,recall,f1,false_alarm_rate,missed_detection_rate
0,0.50,0.3333,0.3333,0.3333,0.6667,0.6667
1,0.55,0.3333,0.3333,0.3333,0.6667,0.6667
2,0.60,0.3333,0.3333,0.3333,0.6667,0.6667
3,0.65,0.3333,0.3333,0.3333,0.6667,0.6667
4,0.70,0.3333,0.3333,0.3333,0.6667,0.6667
5,0.75,0.3333,0.3333,0.3333,0.6667,0.6667
6,0.80,0.3333,0.3333,0.3333,0.6667,0.6667


[paired-harness] Not run on simulated pairs — no fused metric is claimed.


## 12. Persistence, results, and limitations

Persist the policy, artifact provenance, rule tests, real checks, scenario results, and
an explicit "no paired data" status. Existing fusion artifacts are archived before
overwriting.

In [ ]:
# @title 12. Persist policy, provenance, and report
os.makedirs(FUSION_DIR, exist_ok=True)
if any(os.path.exists(path) for path in (
    FUSION_POLICY_PATH, FUSION_META_PATH, FUSION_REPORT_PATH,
    FUSION_SCENARIOS_CSV_PATH, FUSION_SCENARIOS_JSON_PATH,
)):
    os.makedirs(FUSION_HISTORY_DIR, exist_ok=True)
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    for path in (
        FUSION_POLICY_PATH, FUSION_META_PATH, FUSION_REPORT_PATH,
        FUSION_SCENARIOS_CSV_PATH, FUSION_SCENARIOS_JSON_PATH,
    ):
        if os.path.exists(path):
            shutil.copy2(
                path,
                os.path.join(
                    FUSION_HISTORY_DIR,
                    f'{os.path.basename(path)}.{stamp}',
                ),
            )
    print(f'[persist] archived previous fusion artifacts to '
          f'{FUSION_HISTORY_DIR}')

POLICY_PAYLOAD = DEFAULT_POLICY.to_dict()
POLICY_PAYLOAD['name'] = 'smokeguard_gated_weighted_late_fusion'
POLICY_PAYLOAD['created_at_utc'] = utc_now_iso()
POLICY_PAYLOAD['label_semantics'] = {
    'fusion': 'csi_hit AND sensor_hit AND score >= fused_threshold',
    'smoke_only': 'sensor_hit AND NOT fusion (possible alert)',
    'csi_only': (
        'csi_only_enabled AND p_sensor is not None AND p_sensor < sensor gate '
        'AND csi_hit AND p_csi >= csi_only_min_prob (experimental, off by default)'
    ),
    'fused_score': '0.50*p_csi + 0.50*p_sensor; null for csi_only',
}
atomic_write_json(FUSION_POLICY_PATH, POLICY_PAYLOAD)

FUSION_META = {
    'schema_version': FUSION_SCHEMA_VERSION,
    'created_at_utc': utc_now_iso(),
    'policy_path': FUSION_POLICY_PATH,
    'artifacts': {
        'csi_model': {
            'path': CSI_MODEL_PATH,
            'sha256': sha256_file(CSI_MODEL_PATH),
        },
        'csi_metadata': {
            'path': CSI_META_PATH,
            'sha256': sha256_file(CSI_META_PATH),
        },
        'csi_platt': {
            'path': CSI_PLATT_USED_PATH,
            'sha256': sha256_file(CSI_PLATT_USED_PATH),
        },
        'sensor_pipeline': {
            'path': SENSOR_PIPELINE_PATH,
            'sha256': sha256_file(SENSOR_PIPELINE_PATH),
        },
        'sensor_metadata': {
            'path': SENSOR_META_PATH,
            'sha256': sha256_file(SENSOR_META_PATH),
        },
    },
    'fixtures': {
        'csi': {
            'path': CSI_FIXTURE_PATH,
            'sha256': sha256_file(CSI_FIXTURE_PATH),
        },
        'sensor': {
            'path': SENSOR_FIXTURE_PATH,
            'sha256': sha256_file(SENSOR_FIXTURE_PATH),
        },
    },
    'csi_classes': SEVEN_CLASSES,
    'sensor_classes': SENSOR_CLASSES,
    'sensor_threshold': SENSOR_THRESHOLD,
    'sensor_threshold_source': 'smoke_svm.meta.json:threshold.value',
    'labels': ['fusion', 'smoke_only', 'csi_only'],
    'csi_frozen_split_sha256': CSI_META.get('frozen_split_sha256'),
    'sensor_frozen_split_sha256': SENSOR_META.get('frozen_split_sha256'),
    'csi_meta_tag': CSI_META.get('tag'),
    'sensor_meta_tag': SENSOR_META.get('tag'),
    'tensorflow_version': tf.__version__,
    'keras_version': getattr(keras, '__version__', 'unknown'),
    'numpy_version': np.__version__,
    'pandas_version': pd.__version__,
}
atomic_write_json(FUSION_META_PATH, FUSION_META)

FUSION_REPORT = {
    'schema_version': FUSION_SCHEMA_VERSION,
    'created_at_utc': utc_now_iso(),
    'policy': POLICY_PAYLOAD,
    'rule_tests': frame_records(RULE_RESULTS),
    'real_checks': REAL_CHECKS,
    'scenario_input_checks': SCENARIO_INPUT_CHECKS,
    'scenario_summary': frame_records(SCENARIO_ALL),
    'scenario_sensitivity': frame_records(SENSITIVITY_DF),
    'decision_surface_audit': frame_records(AUDIT_DF),
    'paired_data': {
        'available': False,
        'reason': (
            'No synchronized CSI + sensor recordings exist. CSI sessions '
            'are 2026-10-13/15/16/23; the sensor v2 session is 2026-10-27.'
        ),
    },
    'simulated_scenarios': True,
    'fused_metrics': 'not_computed',
    'caveat': (
        'Scenario alert rates are label-conditional simulations from two '
        'different test sessions, not fused accuracy or real-world rates.'
    ),
}
atomic_write_json(FUSION_REPORT_PATH, FUSION_REPORT)

with open(FUSION_POLICY_PATH, 'r', encoding='utf-8') as handle:
    RELOADED_POLICY = json.load(handle)
assert FusionPolicy.from_dict(RELOADED_POLICY) == DEFAULT_POLICY
with open(FUSION_REPORT_PATH, 'r', encoding='utf-8') as handle:
    RELOADED_REPORT = json.load(handle)
assert RELOADED_REPORT['paired_data']['available'] is False
assert RELOADED_REPORT['fused_metrics'] == 'not_computed'
assert RELOADED_REPORT['simulated_scenarios'] is True

print('\n=== Fusion policy ===')
print(json.dumps(POLICY_PAYLOAD, indent=2, sort_keys=True))
print(f'\n[persist] policy:    {FUSION_POLICY_PATH}')
print(f'[persist] metadata:  {FUSION_META_PATH}')
print(f'[persist] report:    {FUSION_REPORT_PATH}')
print(f'[persist] scenarios: {FUSION_SCENARIOS_CSV_PATH}')
print('\nNo fused accuracy/ROC/precision/recall is reported: the two test sets '
      'are not time-synchronized. The scenario tables are simulations.')


[persist] archived previous fusion artifacts to /content/drive/MyDrive/CSI_TRAINING/models/fusion_v1/history

=== Fusion policy ===
{
  "allow_sub_anchor": false,
  "anchor_score": 0.4524994849247028,
  "created_at_utc": "2026-11-08T22:14:28Z",
  "csi_gate_mode": "argmax",
  "csi_gate_threshold": 0.5,
  "csi_only_enabled": false,
  "csi_only_min_prob": 0.0,
  "fused_threshold": 0.7,
  "label_semantics": {
    "csi_only": "csi_only_enabled AND p_sensor is not None AND p_sensor < sensor gate AND csi_hit AND p_csi >= csi_only_min_prob (experimental, off by default)",
    "fused_score": "0.50*p_csi + 0.50*p_sensor; null for csi_only",
    "fusion": "csi_hit AND sensor_hit AND score >= fused_threshold",
    "smoke_only": "sensor_hit AND NOT fusion (possible alert)"
  },
  "name": "smokeguard_gated_weighted_late_fusion",
  "require_both": true,
  "schema_version": 2,
  "sensor_gate_threshold": 0.4049989698494056,
  "weight_csi": 0.5,
  "weight_sensor": 0.5
}

[persist] policy:    /content/dr

## Notes and limitations

- **The conjunction gate is the safety rule.** The weighted sum cannot, by itself,
  keep sensor-only and CSI-only cases silent. A confirmed **`fusion`** requires both
  gates and a fused score of at least `0.70`.
- **Balanced weights.** The deployed policy uses `0.50 CSI / 0.50 sensor`. The smoke
  sensor supplies the confirming gate; CSI supplies the required gesture gate because
  it confuses drinking and similar gestures with smoking.
- **`smoke_only` is a possible alert, not a confirmed fusion.** It fires whenever the
  sensor gate is on but the CSI gate is off or the score is below `0.70`. The backend
  broadcasts it with distinct "possible smoking activity" UI styling.
- **`csi_only` is experimental and off by default.** It fires only when enabled, the
  sensor is operating below its gate, and the CSI calibrated argmax is `smoke`; its
  fused score is `null`.
- **Accepted compromise:** if a person is actually smoking but the sensor does not
  trigger, the system stays silent (unless the experimental csi-only class is enabled).
- **The fused score is a policy score, not a joint posterior.** CSI and sensor
  calibrations come from different sessions and different label semantics.
- **Scenario simulations are not evaluation.** The CSI and sensor held-out sets are
  from different recording dates and are paired by label/outcome. Their label rates
  illustrate the rule and the risk paths, but they are not fused accuracy or
  real-world prevalence.
- **To obtain real fused metrics,** collect synchronized CSI + sensor recordings with
  a shared decision-timestamp table and one interval-level `y_true` definition, then
  run `evaluate_fusion` and `sweep_fused_threshold`.
- The balanced `0.50/0.50` weights and fused threshold `0.70` match the deployed
  backend; they should only be re-tuned on paired validation data. The policy classes
  make that a configuration change, not a code change.
- The offline notebook applies the decision function to fixed held-out windows; it
  does not reproduce the backend's 4–7 s CSI duration scan, 0.2 s evaluation cadence,
  30 s alert cooldown, static-background capture, or SQLite state machine.


## 13. Late-fusion simulation tables for the paper

This section evaluates **late fusion only**. The `smoke_only` and `csi_only`
labels are out of scope: every table counts only the `fusion` decision
(CSI gate **and** sensor gate **and** fused score ≥ threshold).

**Simulation definition.** The held-out sets are unpaired: 224 CSI test
windows (32/class, 7 classes) and 108 sensor test intervals (73 no-smoke,
35 smoke) from different sessions. Every CSI window is paired with every
sensor interval by class/state (224 × 108 = 24,192 simulated pairs). A pair is
**positive** only when the CSI true class is `smoke` **and** the sensor true
state is `smoke` (32 × 35 = 1,120 pairs); every other combination is negative
(23,072 pairs; 4.63% simulated prevalence). This is a simulation, not
synchronized co-occurrence, fused accuracy, or real-world prevalence.

**Deployed reference policy.** 0.50 CSI / 0.50 sensor, fused threshold 0.70,
sensor gate 0.4049989698494056, CSI argmax gate, `csi_only` off.

Primary metrics are **TPR** and **FPR** (and `J = TPR − FPR`), plus the
class-conditional rates. Precision/F1/accuracy depend on the artificial
cross-product prevalence and are included only in the CSV for completeness,
not in the paper-facing tables.


In [ ]:
# @title 13a. Late-fusion pair table and simulation tables
required = [
    'CSI_TEST', 'SENSOR_TEST', 'DEFAULT_POLICY', 'fuse_scores', 'replace',
    'SEVEN_CLASSES', 'SENSOR_CLASSES', 'COMPARISON_EPS', 'FUSION_DIR',
    'CSI_FIXTURE_PATH', 'SENSOR_FIXTURE_PATH', 'CSI_FIXTURE_META_PATH',
    'SENSOR_FIXTURE_META_PATH', 'sha256_file', 'atomic_write_text',
    'atomic_write_json', 'frame_records', 'utc_now_iso',
]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        'Runtime restarted or prerequisite cells were skipped. Rerun: '
        'setup -> preflight -> sensor/CSI adapters -> policy/functions -> '
        '9a (fixture tables), then rerun Section 13. Missing: '
        f'{missing}'
    )

assert len(CSI_TEST) == 224, len(CSI_TEST)
assert len(SENSOR_TEST) == 108, len(SENSOR_TEST)
assert set(CSI_TEST['true_class']) == set(SEVEN_CLASSES)
assert set(SENSOR_TEST['true_state']) == set(SENSOR_CLASSES)
assert int(CSI_TEST['csi_hit'].sum()) == 32, int(CSI_TEST['csi_hit'].sum())
assert int(SENSOR_TEST['sensor_hit'].sum()) == 37, int(SENSOR_TEST['sensor_hit'].sum())

REFERENCE = {
    'weight_csi': 0.50,
    'weight_sensor': 0.50,
    'fused_threshold': 0.70,
    'sensor_gate_threshold': 0.4049989698494056,
    'csi_gate_mode': 'argmax',
    'csi_only_enabled': False,
}
for key, expected in REFERENCE.items():
    actual = getattr(DEFAULT_POLICY, key)
    if isinstance(expected, float):
        assert abs(float(actual) - expected) < 1e-12, (key, actual, expected)
    else:
        assert actual == expected, (key, actual, expected)
assert abs(float(SENSOR_THRESHOLD) - REFERENCE['sensor_gate_threshold']) < 1e-12

csi_table = CSI_TEST.reset_index(drop=True)
sensor_table = SENSOR_TEST.reset_index(drop=True)
n_csi = len(csi_table)
n_sensor = len(sensor_table)
pairs = pd.DataFrame({
    'csi_index': np.repeat(np.arange(n_csi), n_sensor),
    'sensor_index': np.tile(np.arange(n_sensor), n_csi),
    'csi_true_class': np.repeat(csi_table['true_class'].to_numpy(), n_sensor),
    'csi_pred_class': np.repeat(csi_table['pred_class'].to_numpy(), n_sensor),
    'p_csi_smoke': np.repeat(csi_table['p_csi_smoke'].to_numpy(dtype=float), n_sensor),
    'csi_hit': np.repeat(csi_table['csi_hit'].to_numpy(dtype=bool), n_sensor),
    'sensor_true_state': np.tile(sensor_table['true_state'].to_numpy(), n_csi),
    'p_sensor_smoke': np.tile(sensor_table['p_sensor_smoke'].to_numpy(dtype=float), n_csi),
    'sensor_hit': np.tile(sensor_table['sensor_hit'].to_numpy(dtype=bool), n_csi),
})
pairs['y_true_strict'] = (
    (pairs['csi_true_class'] == 'smoke')
    & (pairs['sensor_true_state'] == 'smoke')
)
assert len(pairs) == 224 * 108 == 24192, len(pairs)
assert int(pairs['y_true_strict'].sum()) == 1120
assert int((~pairs['y_true_strict']).sum()) == 23072


def _fusion_alert(pair_frame, policy, csi_hit_override=None):
    p_csi = pair_frame['p_csi_smoke'].to_numpy(dtype=np.float64)
    p_sensor = pair_frame['p_sensor_smoke'].to_numpy(dtype=np.float64)
    if csi_hit_override is None:
        csi_hit = pair_frame['csi_hit'].to_numpy(dtype=bool)
    else:
        csi_hit = np.asarray(csi_hit_override, dtype=bool)
    fused = fuse_scores(p_csi, p_sensor, csi_hit, policy)
    label = fused['label'].to_numpy(dtype=object)
    alert = label == 'fusion'
    explicit = (
        csi_hit
        & (p_sensor >= float(policy.sensor_gate_threshold))
        & ((float(policy.weight_csi) * p_csi
            + float(policy.weight_sensor) * p_sensor)
           >= float(policy.fused_threshold) - COMPARISON_EPS)
    )
    assert np.array_equal(alert, explicit), 'fusion label/mask mismatch'
    return fused, alert


def _metrics(pair_frame, policy, csi_hit_override=None):
    _, alert = _fusion_alert(pair_frame, policy, csi_hit_override)
    truth = pair_frame['y_true_strict'].to_numpy(dtype=bool)
    tp = int(np.sum(alert & truth))
    fp = int(np.sum(alert & ~truth))
    fn = int(np.sum(~alert & truth))
    tn = int(np.sum(~alert & ~truth))
    n_pos = int(truth.sum())
    n_neg = int((~truth).sum())
    tpr = tp / n_pos if n_pos else float('nan')
    fpr = fp / n_neg if n_neg else float('nan')
    precision = tp / (tp + fp) if (tp + fp) else float('nan')
    recall = tpr
    if np.isfinite(precision) and np.isfinite(recall) and (precision + recall):
        f1 = 2.0 * precision * recall / (precision + recall)
    else:
        f1 = float('nan')
    balanced = (tpr + (1.0 - fpr)) / 2.0
    return {
        'n_pos': n_pos, 'n_neg': n_neg,
        'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'tpr': tpr, 'fpr': fpr, 'j_stat': tpr - fpr,
        'precision': precision, 'recall': recall, 'f1': f1,
        'balanced_accuracy': balanced,
        'fusion_alerts': tp + fp,
        'prevalence': n_pos / max(n_pos + n_neg, 1),
    }


# --- Table 1: class-conditional fusion rates (deployed policy) -------------
rate_rows = []
for class_name in SEVEN_CLASSES:
    for sensor_state in SENSOR_CLASSES:
        sub = pairs[
            (pairs['csi_true_class'] == class_name)
            & (pairs['sensor_true_state'] == sensor_state)
        ]
        _, alert = _fusion_alert(sub, DEFAULT_POLICY)
        per_window = (
            sub.assign(_alert=alert)
            .groupby('csi_index')['_alert']
            .mean()
        )
        rate_rows.append({
            'csi_true_class': class_name,
            'sensor_true_state': sensor_state,
            'n_csi': int(sub['csi_index'].nunique()),
            'n_sensor': int(sub['sensor_index'].nunique()),
            'n_pairs': int(len(sub)),
            'fusion_alerts': int(alert.sum()),
            'fusion_rate': float(alert.mean()),
            'per_csi_window_rate_sd': float(per_window.std(ddof=1)),
            'is_true_positive_cell': (
                class_name == 'smoke' and sensor_state == 'smoke'
            ),
        })
T1_LONG = pd.DataFrame(rate_rows)

wide_rows = []
for class_name in SEVEN_CLASSES:
    no_row = T1_LONG[
        (T1_LONG['csi_true_class'] == class_name)
        & (T1_LONG['sensor_true_state'] == 'no-smoke')
    ].iloc[0]
    smoke_row = T1_LONG[
        (T1_LONG['csi_true_class'] == class_name)
        & (T1_LONG['sensor_true_state'] == 'smoke')
    ].iloc[0]
    wide_rows.append({
        'csi_class': class_name,
        'n_no_smoke': int(no_row['n_pairs']),
        'fusion_rate_no_smoke': float(no_row['fusion_rate']),
        'n_smoke': int(smoke_row['n_pairs']),
        'fusion_rate_smoke': float(smoke_row['fusion_rate']),
        'delta': float(smoke_row['fusion_rate'] - no_row['fusion_rate']),
        'rate_sd_smoke': float(smoke_row['per_csi_window_rate_sd']),
        'positive_cell': class_name == 'smoke',
    })
T1_WIDE = pd.DataFrame(wide_rows)

print('Table 1 — class-conditional fusion rates (deployed policy):')
display(T1_LONG.round(3))
print('Table 1 — paper-facing wide table:')
display(T1_WIDE.round(3))

# --- Table 2: weights x fused threshold ------------------------------------
weight_rows = []
for weight_csi, weight_sensor in ((0.50, 0.50), (0.40, 0.60), (0.30, 0.70)):
    for fused_threshold in (0.60, 0.65, 0.70, 0.75):
        policy = replace(
            DEFAULT_POLICY,
            weight_csi=weight_csi,
            weight_sensor=weight_sensor,
            fused_threshold=fused_threshold,
        )
        row = _metrics(pairs, policy)
        row.update({
            'weight_csi': weight_csi,
            'weight_sensor': weight_sensor,
            'fused_threshold': fused_threshold,
            'is_deployed': (
                weight_csi == 0.50 and fused_threshold == 0.70
            ),
        })
        weight_rows.append(row)
T2 = pd.DataFrame(weight_rows)[[
    'weight_csi', 'weight_sensor', 'fused_threshold',
    'n_pos', 'n_neg', 'tp', 'fp', 'fn', 'tn',
    'tpr', 'fpr', 'j_stat', 'precision', 'recall', 'f1',
    'balanced_accuracy', 'fusion_alerts', 'is_deployed',
]]
print('Table 2 — weight x fused-threshold sweep:')
display(T2.round(3))

# --- Table 3: sensor-gate sweep --------------------------------------------
# Lower gates show the trade-off; above ~0.45 the 35 smoke intervals all
# clear the gate, so the rates flatten.
sensor_gates = [
    0.30, 0.35, 0.40, 0.4049989698494056,
    0.45, 0.50, 0.55, 0.60,
]
sensor_rows = []
for sensor_gate in sensor_gates:
    policy = replace(DEFAULT_POLICY, sensor_gate_threshold=sensor_gate)
    row = _metrics(pairs, policy)
    row.update({
        'sensor_gate': sensor_gate,
        'n_sensor_hit': int(
            (SENSOR_TEST['p_sensor_smoke'] >= sensor_gate).sum()
        ),
        'n_sensor_total': int(len(SENSOR_TEST)),
        'is_deployed': abs(sensor_gate - 0.4049989698494056) < 1e-12,
    })
    sensor_rows.append(row)
T3 = pd.DataFrame(sensor_rows)[[
    'sensor_gate', 'n_sensor_hit', 'n_sensor_total',
    'n_pos', 'n_neg', 'tp', 'fp', 'fn', 'tn',
    'tpr', 'fpr', 'j_stat', 'precision', 'recall', 'f1',
    'balanced_accuracy', 'fusion_alerts', 'is_deployed',
]]
assert int(T3.loc[T3['is_deployed'], 'n_sensor_hit'].iloc[0]) == 37
print('Table 3 — sensor-gate sweep:')
display(T3.round(3))

# --- Table 4: CSI gate mode/threshold sweep --------------------------------
csi_rows = []
argmax_row = _metrics(pairs, DEFAULT_POLICY)
argmax_row.update({
    'csi_gate_mode': 'argmax',
    'csi_gate_threshold': float('nan'),
    'csi_gate_threshold_display': '—',
    'n_csi_hit': int(CSI_TEST['csi_hit'].sum()),
    'n_csi_total': int(len(CSI_TEST)),
    'is_deployed': True,
})
csi_rows.append(argmax_row)
for threshold in (0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90):
    hit = (pairs['p_csi_smoke'] >= threshold).to_numpy(dtype=bool)
    policy = replace(
        DEFAULT_POLICY,
        csi_gate_mode='probability',
        csi_gate_threshold=threshold,
        allow_sub_anchor=True,
    )
    row = _metrics(pairs, policy, csi_hit_override=hit)
    row.update({
        'csi_gate_mode': 'probability',
        'csi_gate_threshold': threshold,
        'csi_gate_threshold_display': f'{threshold:.2f}',
        'n_csi_hit': int((CSI_TEST['p_csi_smoke'] >= threshold).sum()),
        'n_csi_total': int(len(CSI_TEST)),
        'is_deployed': False,
    })
    csi_rows.append(row)
T4 = pd.DataFrame(csi_rows)[[
    'csi_gate_mode', 'csi_gate_threshold', 'csi_gate_threshold_display',
    'n_csi_hit', 'n_csi_total',
    'n_pos', 'n_neg', 'tp', 'fp', 'fn', 'tn',
    'tpr', 'fpr', 'j_stat', 'precision', 'recall', 'f1',
    'balanced_accuracy', 'fusion_alerts', 'is_deployed',
]]
print('Table 4 — CSI gate mode/threshold sweep:')
display(T4.round(3))

# --- Table 5: fusion confusion at the deployed policy ----------------------
DEPLOYED_METRICS = _metrics(pairs, DEFAULT_POLICY)
T5_COUNTS = pd.DataFrame([
    {'simulated_truth': 'positive', 'fusion_alert': True,
     'count': DEPLOYED_METRICS['tp']},
    {'simulated_truth': 'positive', 'fusion_alert': False,
     'count': DEPLOYED_METRICS['fn']},
    {'simulated_truth': 'negative', 'fusion_alert': True,
     'count': DEPLOYED_METRICS['fp']},
    {'simulated_truth': 'negative', 'fusion_alert': False,
     'count': DEPLOYED_METRICS['tn']},
])
T5_WIDE = pd.DataFrame({
    'simulated_truth': ['positive', 'negative'],
    'fusion_alert': [DEPLOYED_METRICS['tp'], DEPLOYED_METRICS['fp']],
    'no_fusion_alert': [DEPLOYED_METRICS['fn'], DEPLOYED_METRICS['tn']],
})
T5_METRICS = pd.DataFrame([{
    'n_pos': DEPLOYED_METRICS['n_pos'],
    'n_neg': DEPLOYED_METRICS['n_neg'],
    'prevalence': DEPLOYED_METRICS['prevalence'],
    'tp': DEPLOYED_METRICS['tp'],
    'fp': DEPLOYED_METRICS['fp'],
    'fn': DEPLOYED_METRICS['fn'],
    'tn': DEPLOYED_METRICS['tn'],
    'tpr': DEPLOYED_METRICS['tpr'],
    'fpr': DEPLOYED_METRICS['fpr'],
    'j_stat': DEPLOYED_METRICS['j_stat'],
    'precision': DEPLOYED_METRICS['precision'],
    'recall': DEPLOYED_METRICS['recall'],
    'f1': DEPLOYED_METRICS['f1'],
    'balanced_accuracy': DEPLOYED_METRICS['balanced_accuracy'],
    'fusion_alerts': DEPLOYED_METRICS['fusion_alerts'],
}])
print('Table 5 — fusion confusion at the deployed policy:')
display(T5_COUNTS)
display(T5_WIDE)
display(T5_METRICS.round(3))

# --- Validation ------------------------------------------------------------
assert len(T1_LONG) == 14
assert int(T1_LONG['n_pairs'].sum()) == 24192
assert int(T1_LONG['is_true_positive_cell'].sum()) == 1
assert len(T2) == 12 and int(T2['is_deployed'].sum()) == 1
assert len(T3) == 8 and int(T3['is_deployed'].sum()) == 1
assert len(T4) == 8 and int(T4['is_deployed'].sum()) == 1
deployed_rows = [
    T2.loc[T2['is_deployed']].iloc[0],
    T3.loc[T3['is_deployed']].iloc[0],
    T4.loc[T4['is_deployed']].iloc[0],
]
for row in deployed_rows:
    for key in ('tp', 'fp', 'fn', 'tn', 'tpr', 'fpr'):
        assert abs(float(row[key]) - float(DEPLOYED_METRICS[key])) < 1e-12
assert int(T5_COUNTS['count'].sum()) == 24192
assert int(T5_WIDE[['fusion_alert', 'no_fusion_alert']].to_numpy().sum()) == 24192
assert int(DEPLOYED_METRICS['tp']) + int(DEPLOYED_METRICS['fn']) == 1120
assert int(DEPLOYED_METRICS['fp']) + int(DEPLOYED_METRICS['tn']) == 23072
for frame in (T1_LONG, T1_WIDE, T2, T3, T4, T5_COUNTS, T5_WIDE, T5_METRICS):
    assert not any(
        'smoke_only' in str(col) or 'csi_only' in str(col)
        for col in frame.columns
    )
print('\n[13a] pair table and all five tables validated.')
print(f'[13a] deployed: TP={DEPLOYED_METRICS["tp"]}, FP={DEPLOYED_METRICS["fp"]}, '
      f'FN={DEPLOYED_METRICS["fn"]}, TN={DEPLOYED_METRICS["tn"]}, '
      f'TPR={DEPLOYED_METRICS["tpr"]:.3f}, FPR={DEPLOYED_METRICS["fpr"]:.3f}, '
      f'J={DEPLOYED_METRICS["j_stat"]:.3f}')


Table 1 — class-conditional fusion rates (deployed policy):


,csi_true_class,sensor_true_state,n_csi,n_sensor,n_pairs,fusion_alerts,fusion_rate,per_csi_window_rate_sd,is_true_positive_cell
0,smoke,no-smoke,32,73,2336,0,0.000,0.000,False
1,smoke,smoke,32,35,1120,840,0.750,0.440,True
2,eat,no-smoke,32,73,2336,0,0.000,0.000,False
3,eat,smoke,32,35,1120,0,0.000,0.000,False
4,drink,no-smoke,32,73,2336,0,0.000,0.000,False
5,drink,smoke,32,35,1120,105,0.094,0.296,False
6,sit,no-smoke,32,73,2336,0,0.000,0.000,False
7,sit,smoke,32,35,1120,0,0.000,0.000,False
8,stand,no-smoke,32,73,2336,0,0.000,0.000,False
9,stand,smoke,32,35,1120,0,0.000,0.000,False


Table 1 — paper-facing wide table:


,csi_class,n_no_smoke,fusion_rate_no_smoke,n_smoke,fusion_rate_smoke,delta,rate_sd_smoke,positive_cell
0,smoke,2336,0.0,1120,0.750,0.750,0.440,True
1,eat,2336,0.0,1120,0.000,0.000,0.000,False
2,drink,2336,0.0,1120,0.094,0.094,0.296,False
3,sit,2336,0.0,1120,0.000,0.000,0.000,False
4,stand,2336,0.0,1120,0.000,0.000,0.000,False
5,random,2336,0.0,1120,0.031,0.031,0.177,False
6,static,2336,0.0,1120,0.031,0.031,0.177,False


Table 2 — weight x fused-threshold sweep:


,weight_csi,weight_sensor,fused_threshold,n_pos,n_neg,tp,fp,fn,tn,tpr,fpr,j_stat,precision,recall,f1,balanced_accuracy,fusion_alerts,is_deployed
0,0.5,0.5,0.60,1120,23072,874,291,246,22781,0.780,0.013,0.768,0.750,0.780,0.765,0.884,1165,False
1,0.5,0.5,0.65,1120,23072,843,231,277,22841,0.753,0.010,0.743,0.785,0.753,0.768,0.871,1074,False
2,0.5,0.5,0.70,1120,23072,840,175,280,22897,0.750,0.008,0.742,0.828,0.750,0.787,0.871,1015,True
3,0.5,0.5,0.75,1120,23072,839,175,281,22897,0.749,0.008,0.742,0.827,0.749,0.786,0.871,1014,False
4,0.4,0.6,0.60,1120,23072,875,289,245,22783,0.781,0.013,0.769,0.752,0.781,0.766,0.884,1164,False
5,0.4,0.6,0.65,1120,23072,874,243,246,22829,0.780,0.011,0.770,0.782,0.780,0.781,0.885,1117,False
6,0.4,0.6,0.70,1120,23072,860,211,260,22861,0.768,0.009,0.759,0.803,0.768,0.785,0.879,1071,False
7,0.4,0.6,0.75,1120,23072,840,175,280,22897,0.750,0.008,0.742,0.828,0.750,0.787,0.871,1015,False
8,0.3,0.7,0.60,1120,23072,875,245,245,22827,0.781,0.011,0.771,0.781,0.781,0.781,0.885,1120,False
9,0.3,0.7,0.65,1120,23072,875,245,245,22827,0.781,0.011,0.771,0.781,0.781,0.781,0.885,1120,False


... (5 outputs left out)

,simulated_truth,fusion_alert,count
0,positive,True,840
1,positive,False,280
2,negative,True,175
3,negative,False,22897


,simulated_truth,fusion_alert,no_fusion_alert
0,positive,840,280
1,negative,175,22897


,n_pos,n_neg,prevalence,tp,fp,fn,tn,tpr,fpr,j_stat,precision,recall,f1,balanced_accuracy,fusion_alerts
0,1120,23072,0.046,840,175,280,22897,0.75,0.008,0.742,0.828,0.75,0.787,0.871,1015



[13a] pair table and all five tables validated.
[13a] deployed: TP=840, FP=175, FN=280, TN=22897, TPR=0.750, FPR=0.008, J=0.742


In [ ]:
# @title 13b. Export paper tables (CSV + LaTeX + metadata)
PAPER_TABLES_DIR = os.path.join(FUSION_DIR, 'paper_tables')
PAPER_TABLES_HISTORY_DIR = os.path.join(PAPER_TABLES_DIR, 'history')
os.makedirs(PAPER_TABLES_DIR, exist_ok=True)
os.makedirs(PAPER_TABLES_HISTORY_DIR, exist_ok=True)

_existing_paper_files = [
    name for name in os.listdir(PAPER_TABLES_DIR)
    if os.path.isfile(os.path.join(PAPER_TABLES_DIR, name))
    and (name.startswith('table_') or name == 'paper_tables_meta.json')
]
if _existing_paper_files:
    _paper_stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    _paper_history = os.path.join(PAPER_TABLES_HISTORY_DIR, _paper_stamp)
    os.makedirs(_paper_history, exist_ok=True)
    for name in _existing_paper_files:
        shutil.copy2(
            os.path.join(PAPER_TABLES_DIR, name),
            os.path.join(_paper_history, name),
        )
    print(f'[paper] archived {len(_existing_paper_files)} old file(s) to '
          f'{_paper_history}')

CSI_FIXTURE_SHA = sha256_file(CSI_FIXTURE_PATH)
SENSOR_FIXTURE_SHA = sha256_file(SENSOR_FIXTURE_PATH)
CSI_FIXTURE_META_SHA = sha256_file(CSI_FIXTURE_META_PATH)
SENSOR_FIXTURE_META_SHA = sha256_file(SENSOR_FIXTURE_META_PATH)
POLICY_FINGERPRINT = hashlib.sha256(
    json.dumps(
        DEFAULT_POLICY.to_dict(), sort_keys=True, separators=(',', ':')
    ).encode('utf-8')
).hexdigest()

T1_CAPTION = (
    'Simulated class-conditional late-fusion rates on the held-out fixtures. '
    'CSI windows (n = 224; 32 per class) and sensor intervals (n = 108; '
    '73 no-smoke, 35 smoke) are from different sessions and are paired by '
    'class and state, not synchronized. Each cell is the fraction of '
    'cross-product pairs in which the deployed policy (CSI 0.50 / sensor '
    '0.50, fused threshold 0.70, sensor gate 0.404999, CSI argmax) emitted a '
    'fusion alert; N is the number of pairs and rate = alerts/N. The smoke '
    'row\'s Fusion rate (smoke) cell (Positive cell = Yes) is the only '
    'simulated true-positive cell. Only fusion decisions count; smoke-only '
    'and CSI-only are out of scope. These are simulations, not real-world '
    'fused accuracy. Fixture SHA-256: CSI '
    f'{CSI_FIXTURE_SHA[:12]}, sensor {SENSOR_FIXTURE_SHA[:12]}; policy '
    f'fingerprint {POLICY_FINGERPRINT[:12]}.'
)
T2_CAPTION = (
    'Simulated weight x fused-threshold sensitivity. Positive = CSI true '
    'class smoke AND sensor true state smoke (1,120/24,192 pairs; 4.63% '
    'simulated prevalence); all other class/state pairs are negative. '
    'TPR = TP/(TP+FN), FPR = FP/(FP+TN), J = TPR - FPR. Precision/F1/'
    'balanced accuracy are prevalence-dependent and are provided only in '
    'the CSV, not in this paper table. The deployed row (0.50/0.50, '
    'threshold 0.70) is marked Deployed = Yes. No synchronized paired '
    'recordings exist; these are simulation results, not paired-data '
    'performance. Fixture SHA-256: CSI '
    f'{CSI_FIXTURE_SHA[:12]}, sensor {SENSOR_FIXTURE_SHA[:12]}; policy '
    f'fingerprint {POLICY_FINGERPRINT[:12]}.'
)
T3_CAPTION = (
    'Simulated sensor-gate sweep at the deployed weights and fused threshold '
    '(0.50/0.50, threshold 0.70). Sensor hits is the number of the 108 '
    'held-out sensor intervals with P(smoke) >= sensor gate. TPR/FPR use '
    'strict-AND simulated truth over the 224 x 108 cross-product. The '
    'deployed sensor gate 0.404999 (SVM F1-max out-of-fold threshold) is '
    'marked Deployed = Yes. Simulated pairings, not real co-occurrence; '
    f'fixture SHA-256: CSI {CSI_FIXTURE_SHA[:12]}, sensor '
    f'{SENSOR_FIXTURE_SHA[:12]}; policy fingerprint {POLICY_FINGERPRINT[:12]}.'
)
T4_CAPTION = (
    'Simulated CSI-gate sweep at the deployed weights and fused threshold '
    '(0.50/0.50, threshold 0.70). Argmax is the deployed gate; probability '
    'rows use P(smoke) >= threshold. CSI hits is the number of the 224 '
    'held-out CSI windows passing the gate. TPR/FPR use strict-AND simulated '
    'truth over the cross-product. The deployed argmax gate is marked '
    'Deployed = Yes. Simulated pairings, not real co-occurrence; fixture '
    f'SHA-256: CSI {CSI_FIXTURE_SHA[:12]}, sensor {SENSOR_FIXTURE_SHA[:12]}; '
    f'policy fingerprint {POLICY_FINGERPRINT[:12]}.'
)
T5_CAPTION = (
    'Simulated late-fusion confusion at the deployed policy (0.50/0.50, '
    'threshold 0.70, sensor gate 0.404999, CSI argmax). Positive = CSI true '
    'class smoke AND sensor true state smoke (1,120/24,192; 4.63% simulated '
    'prevalence); negative = the remaining 23,072 pairs. Only fusion '
    'decisions alert; smoke-only and CSI-only are excluded. TPR and FPR are '
    'conditional and defensible; precision, F1, and balanced accuracy depend '
    'on the artificial prevalence and are reported only for completeness in '
    'the metrics table. No synchronized paired recordings exist; this is not '
    'real-world fused accuracy or ROC. Fixture SHA-256: CSI '
    f'{CSI_FIXTURE_SHA[:12]}, sensor {SENSOR_FIXTURE_SHA[:12]}; policy '
    f'fingerprint {POLICY_FINGERPRINT[:12]}.'
)
T5M_CAPTION = (
    'Simulated late-fusion metrics at the deployed policy. Positive = CSI '
    'true class smoke AND sensor true state smoke (1,120/24,192; 4.63% '
    'simulated prevalence); negative = the remaining 23,072 pairs. TPR and '
    'FPR are conditional; precision, F1, and balanced accuracy are shown for '
    'completeness but depend on the artificial prevalence. These are '
    'simulated cross-product pairings, not synchronized co-occurrence or '
    'real-world fused accuracy. Fixture SHA-256: CSI '
    f'{CSI_FIXTURE_SHA[:12]}, sensor {SENSOR_FIXTURE_SHA[:12]}; policy '
    f'fingerprint {POLICY_FINGERPRINT[:12]}.'
)


def _display_frame(frame, columns, names):
    out = frame[columns].copy()
    for flag in ('is_deployed', 'positive_cell', 'is_true_positive_cell'):
        if flag in out.columns:
            out[flag] = out[flag].map(lambda value: 'Yes' if bool(value) else '')
    for column in out.select_dtypes(include=[np.floating]).columns:
        out[column] = out[column].map(
            lambda value: f'{value:.3f}' if pd.notna(value) else '—'
        )
    return out.rename(columns=names)


def _write_paper_table(frame, base_name, caption, label,
                       paper_columns, display_names):
    csv_frame = frame.copy()
    csv_text = csv_frame.to_csv(index=False, float_format='%.3f')
    csv_path = os.path.join(PAPER_TABLES_DIR, f'{base_name}.csv')
    csv_sha = atomic_write_text(csv_path, csv_text)

    paper = _display_frame(frame, paper_columns, display_names)
    try:
        body = paper.to_latex(
            index=False, escape=False, float_format='%.3f', booktabs=True
        )
    except Exception:
        header = ' & '.join(str(column) for column in paper.columns) + r' \\'
        rows = [
            ' & '.join(str(value) for value in row) + r' \\'
            for row in paper.itertuples(index=False, name=None)
        ]
        body = (
            '\\begin{tabular}{' + 'l' * len(paper.columns) + '}\n'
            '\\toprule\n' + header + '\n\\midrule\n'
            + '\n'.join(rows) + '\n\\bottomrule\n\\end{tabular}\n'
        )
    safe_caption = caption.replace('%', r'\%')
    tex = (
        '\\begin{table}[htbp]\n\\centering\n'
        f'\\caption{{{safe_caption}}}\n'
        f'\\label{{{label}}}\n'
        + body + '\\end{table}\n'
    )
    tex_path = os.path.join(PAPER_TABLES_DIR, f'{base_name}.tex')
    tex_sha = atomic_write_text(tex_path, tex)
    display(paper)
    return {
        'csv_path': csv_path,
        'csv_sha256': csv_sha,
        'tex_path': tex_path,
        'tex_sha256': tex_sha,
        'rows': int(len(frame)),
        'caption': caption,
        'label': label,
        'paper_columns': list(paper.columns),
    }


EXPORTED_TABLES = {}

EXPORTED_TABLES['table_1_fusion_by_class_state_wide'] = _write_paper_table(
    T1_WIDE,
    'table_1_fusion_by_class_state_wide',
    T1_CAPTION,
    'tab:fusion_by_class_state',
    ['csi_class', 'n_no_smoke', 'fusion_rate_no_smoke',
     'n_smoke', 'fusion_rate_smoke', 'delta', 'rate_sd_smoke',
     'positive_cell'],
    {
        'csi_class': 'CSI class',
        'n_no_smoke': 'N (no-smoke)',
        'fusion_rate_no_smoke': 'Fusion rate (no-smoke)',
        'n_smoke': 'N (smoke)',
        'fusion_rate_smoke': 'Fusion rate (smoke)',
        'delta': 'Delta (smoke - no-smoke)',
        'rate_sd_smoke': 'Rate SD (smoke)',
        'positive_cell': 'Positive cell',
    },
)
EXPORTED_TABLES['table_1_fusion_by_class_state_long'] = _write_paper_table(
    T1_LONG,
    'table_1_fusion_by_class_state_long',
    T1_CAPTION,
    'tab:fusion_by_class_state_long',
    ['csi_true_class', 'sensor_true_state', 'n_pairs', 'fusion_alerts',
     'fusion_rate', 'per_csi_window_rate_sd', 'is_true_positive_cell'],
    {
        'csi_true_class': 'CSI class',
        'sensor_true_state': 'Sensor state',
        'n_pairs': 'Pairs',
        'fusion_alerts': 'Fusion alerts',
        'fusion_rate': 'Fusion rate',
        'per_csi_window_rate_sd': 'Rate SD',
        'is_true_positive_cell': 'Positive cell',
    },
)
EXPORTED_TABLES['table_2_weight_threshold_sweep'] = _write_paper_table(
    T2,
    'table_2_weight_threshold_sweep',
    T2_CAPTION,
    'tab:fusion_weight_threshold',
    ['weight_csi', 'weight_sensor', 'fused_threshold',
     'tpr', 'fpr', 'j_stat', 'fusion_alerts', 'is_deployed'],
    {
        'weight_csi': 'CSI weight',
        'weight_sensor': 'Sensor weight',
        'fused_threshold': 'Fused threshold',
        'tpr': 'TPR',
        'fpr': 'FPR',
        'j_stat': 'J',
        'fusion_alerts': 'Fusion alerts',
        'is_deployed': 'Deployed',
    },
)
EXPORTED_TABLES['table_3_sensor_gate_sweep'] = _write_paper_table(
    T3,
    'table_3_sensor_gate_sweep',
    T3_CAPTION,
    'tab:fusion_sensor_gate',
    ['sensor_gate', 'n_sensor_hit', 'n_sensor_total',
     'tpr', 'fpr', 'j_stat', 'fusion_alerts', 'is_deployed'],
    {
        'sensor_gate': 'Sensor gate',
        'n_sensor_hit': 'Sensor hits',
        'n_sensor_total': 'N sensor',
        'tpr': 'TPR',
        'fpr': 'FPR',
        'j_stat': 'J',
        'fusion_alerts': 'Fusion alerts',
        'is_deployed': 'Deployed',
    },
)
EXPORTED_TABLES['table_4_csi_gate_sweep'] = _write_paper_table(
    T4,
    'table_4_csi_gate_sweep',
    T4_CAPTION,
    'tab:fusion_csi_gate',
    ['csi_gate_mode', 'csi_gate_threshold_display',
     'n_csi_hit', 'n_csi_total',
     'tpr', 'fpr', 'j_stat', 'fusion_alerts', 'is_deployed'],
    {
        'csi_gate_mode': 'CSI gate',
        'csi_gate_threshold_display': 'Threshold',
        'n_csi_hit': 'CSI hits',
        'n_csi_total': 'N CSI',
        'tpr': 'TPR',
        'fpr': 'FPR',
        'j_stat': 'J',
        'fusion_alerts': 'Fusion alerts',
        'is_deployed': 'Deployed',
    },
)
EXPORTED_TABLES['table_5_fusion_confusion'] = _write_paper_table(
    T5_WIDE,
    'table_5_fusion_confusion',
    T5_CAPTION,
    'tab:fusion_confusion',
    ['simulated_truth', 'fusion_alert', 'no_fusion_alert'],
    {
        'simulated_truth': 'Simulated truth',
        'fusion_alert': 'Fusion alert',
        'no_fusion_alert': 'No fusion alert',
    },
)
EXPORTED_TABLES['table_5_fusion_metrics'] = _write_paper_table(
    T5_METRICS,
    'table_5_fusion_metrics',
    T5M_CAPTION,
    'tab:fusion_metrics',
    ['n_pos', 'n_neg', 'prevalence',
     'tpr', 'fpr', 'j_stat', 'precision', 'recall', 'f1',
     'balanced_accuracy', 'tp', 'fp', 'fn', 'tn', 'fusion_alerts'],
    {
        'n_pos': 'N positive',
        'n_neg': 'N negative',
        'prevalence': 'Simulated prevalence',
        'tpr': 'TPR',
        'fpr': 'FPR',
        'j_stat': 'J',
        'precision': 'Precision',
        'recall': 'Recall',
        'f1': 'F1',
        'balanced_accuracy': 'Balanced accuracy',
        'tp': 'TP',
        'fp': 'FP',
        'fn': 'FN',
        'tn': 'TN',
        'fusion_alerts': 'Fusion alerts',
    },
)

PAPER_TABLES_META = {
    'schema_version': 1,
    'created_at_utc': utc_now_iso(),
    'simulated': True,
    'paired_data_available': False,
    'evaluation_type': 'cross_product_label_state_simulation',
    'scope': {
        'included': ['fusion'],
        'excluded': ['smoke_only', 'csi_only'],
        'note': (
            'Only fusion decisions (CSI gate AND sensor gate AND fused '
            'score >= threshold) are counted; sensor-only and CSI-only '
            'alerts are out of scope.'
        ),
    },
    'ground_truth': {
        'primary': 'CSI true class == smoke AND sensor true state == smoke',
        'positive_pairs': 1120,
        'negative_pairs': 23072,
        'total_pairs': 24192,
        'simulated_prevalence': 1120 / 24192,
        'alternatives_not_used': [
            'sensor true state == smoke only',
            'CSI true class == smoke only',
        ],
    },
    'policy': DEFAULT_POLICY.to_dict(),
    'policy_fingerprint_sha256': POLICY_FINGERPRINT,
    'fixtures': {
        'csi': {
            'path': CSI_FIXTURE_PATH,
            'sha256': CSI_FIXTURE_SHA,
            'metadata_path': CSI_FIXTURE_META_PATH,
            'metadata_sha256': CSI_FIXTURE_META_SHA,
        },
        'sensor': {
            'path': SENSOR_FIXTURE_PATH,
            'sha256': SENSOR_FIXTURE_SHA,
            'metadata_path': SENSOR_FIXTURE_META_PATH,
            'metadata_sha256': SENSOR_FIXTURE_META_SHA,
        },
    },
    'deployed_metrics': DEPLOYED_METRICS,
    'tables': EXPORTED_TABLES,
    'validation': {
        'pair_table_rows': int(len(pairs)),
        'positive_pairs': int(pairs['y_true_strict'].sum()),
        'negative_pairs': int((~pairs['y_true_strict']).sum()),
        'sensor_hits_at_deployed_gate': int(
            (SENSOR_TEST['p_sensor_smoke'] >= SENSOR_THRESHOLD).sum()
        ),
        'csi_hits_at_argmax': int(CSI_TEST['csi_hit'].sum()),
        'deployed_row_consistent_across_tables': True,
        'smoke_only_or_csi_only_columns': False,
    },
    'caveat': (
        'CSI and sensor held-out sets are from different sessions and are '
        'paired by class/state in a cross-product. These tables are '
        'simulations, not synchronized co-occurrence, fused accuracy, ROC, '
        'or real-world prevalence. TPR/FPR and per-class rates are the '
        'defensible metrics; precision/F1/balanced accuracy depend on the '
        'artificial 4.63% pair prevalence.'
    ),
}
PAPER_TABLES_META_PATH = os.path.join(
    PAPER_TABLES_DIR, 'paper_tables_meta.json'
)
PAPER_TABLES_META_SHA = atomic_write_json(
    PAPER_TABLES_META_PATH, PAPER_TABLES_META
)

# --- Verify exported files -------------------------------------------------
_export_rows = []
for name, info in EXPORTED_TABLES.items():
    reloaded = pd.read_csv(info['csv_path'])
    assert len(reloaded) == info['rows'], (name, len(reloaded), info['rows'])
    assert os.path.exists(info['tex_path'])
    tex_text = open(info['tex_path'], 'r', encoding='utf-8').read()
    assert '\\begin{table}' in tex_text and '\\caption' in tex_text
    assert '\\label' in tex_text
    assert sha256_file(info['csv_path']) == info['csv_sha256']
    assert sha256_file(info['tex_path']) == info['tex_sha256']
    _export_rows.append({
        'table': name,
        'csv': os.path.basename(info['csv_path']),
        'tex': os.path.basename(info['tex_path']),
        'rows': info['rows'],
        'csv_sha256': info['csv_sha256'][:16],
        'tex_sha256': info['tex_sha256'][:16],
    })
with open(PAPER_TABLES_META_PATH, 'r', encoding='utf-8') as _handle:
    _reloaded_meta = json.load(_handle)
assert _reloaded_meta['simulated'] is True
assert _reloaded_meta['paired_data_available'] is False
assert _reloaded_meta['validation']['pair_table_rows'] == 24192
assert sha256_file(PAPER_TABLES_META_PATH) == PAPER_TABLES_META_SHA

print(f'[13b] wrote {len(EXPORTED_TABLES)} table pairs to {PAPER_TABLES_DIR}')
print(f'[13b] metadata: {PAPER_TABLES_META_PATH}')
display(pd.DataFrame(_export_rows))
print('\nPaper-facing files:')
for name, info in EXPORTED_TABLES.items():
    print(f'  {os.path.basename(info["tex_path"])}  ({info["rows"]} rows)')


[paper] archived 15 old file(s) to /content/drive/MyDrive/CSI_TRAINING/models/fusion_v1/paper_tables/history/20261109T100340Z


,CSI class,N (no-smoke),Fusion rate (no-smoke),N (smoke),Fusion rate (smoke),Delta (smoke - no-smoke),Rate SD (smoke),Positive cell
0,smoke,2336,0.000,1120,0.750,0.750,0.440,Yes
1,eat,2336,0.000,1120,0.000,0.000,0.000,
2,drink,2336,0.000,1120,0.094,0.094,0.296,
3,sit,2336,0.000,1120,0.000,0.000,0.000,
4,stand,2336,0.000,1120,0.000,0.000,0.000,
5,random,2336,0.000,1120,0.031,0.031,0.177,
6,static,2336,0.000,1120,0.031,0.031,0.177,


,CSI class,Sensor state,Pairs,Fusion alerts,Fusion rate,Rate SD,Positive cell
0,smoke,no-smoke,2336,0,0.000,0.000,
1,smoke,smoke,1120,840,0.750,0.440,Yes
2,eat,no-smoke,2336,0,0.000,0.000,
3,eat,smoke,1120,0,0.000,0.000,
4,drink,no-smoke,2336,0,0.000,0.000,
5,drink,smoke,1120,105,0.094,0.296,
6,sit,no-smoke,2336,0,0.000,0.000,
7,sit,smoke,1120,0,0.000,0.000,
8,stand,no-smoke,2336,0,0.000,0.000,
9,stand,smoke,1120,0,0.000,0.000,


,CSI weight,Sensor weight,Fused threshold,TPR,FPR,J,Fusion alerts,Deployed
0,0.500,0.500,0.600,0.780,0.013,0.768,1165,
1,0.500,0.500,0.650,0.753,0.010,0.743,1074,
2,0.500,0.500,0.700,0.750,0.008,0.742,1015,Yes
3,0.500,0.500,0.750,0.749,0.008,0.742,1014,
4,0.400,0.600,0.600,0.781,0.013,0.769,1164,
5,0.400,0.600,0.650,0.780,0.011,0.770,1117,
6,0.400,0.600,0.700,0.768,0.009,0.759,1071,
7,0.400,0.600,0.750,0.750,0.008,0.742,1015,
8,0.300,0.700,0.600,0.781,0.011,0.771,1120,
9,0.300,0.700,0.650,0.781,0.011,0.771,1120,


,Sensor gate,Sensor hits,N sensor,TPR,FPR,J,Fusion alerts,Deployed
0,0.300,37,108,0.750,0.008,0.742,1015,
1,0.350,37,108,0.750,0.008,0.742,1015,
2,0.400,37,108,0.750,0.008,0.742,1015,
3,0.405,37,108,0.750,0.008,0.742,1015,Yes
4,0.450,35,108,0.750,0.008,0.742,1015,
5,0.500,35,108,0.750,0.008,0.742,1015,
6,0.550,35,108,0.750,0.008,0.742,1015,
7,0.600,35,108,0.750,0.008,0.742,1015,


,CSI gate,Threshold,CSI hits,N CSI,TPR,FPR,J,Fusion alerts,Deployed
0,argmax,—,32,224,0.750,0.008,0.742,1015,Yes
1,probability,0.30,32,224,0.750,0.008,0.742,1015,
2,probability,0.40,29,224,0.750,0.008,0.742,1015,
3,probability,0.50,29,224,0.750,0.008,0.742,1015,
4,probability,0.60,29,224,0.750,0.008,0.742,1015,
5,probability,0.70,27,224,0.688,0.008,0.680,945,
6,probability,0.80,24,224,0.594,0.008,0.586,840,
7,probability,0.90,10,224,0.281,0.002,0.280,350,


,Simulated truth,Fusion alert,No fusion alert
0,positive,840,280
1,negative,175,22897


,N positive,N negative,Simulated prevalence,TPR,FPR,J,Precision,Recall,F1,Balanced accuracy,TP,FP,FN,TN,Fusion alerts
0,1120,23072,0.046,0.750,0.008,0.742,0.828,0.750,0.787,0.871,840,175,280,22897,1015


[13b] wrote 7 table pairs to /content/drive/MyDrive/CSI_TRAINING/models/fusion_v1/paper_tables
[13b] metadata: /content/drive/MyDrive/CSI_TRAINING/models/fusion_v1/paper_tables/paper_tables_meta.json


,table,csv,tex,rows,csv_sha256,tex_sha256
0,table_1_fusion_by_class_state_wide,table_1_fusion_by_class_state_wide.csv,table_1_fusion_by_class_state_wide.tex,7,e18256bfce1f40fc,1ab5fc5c4b890aef
1,table_1_fusion_by_class_state_long,table_1_fusion_by_class_state_long.csv,table_1_fusion_by_class_state_long.tex,14,dae10149ce769607,437a9e27543d7978
2,table_2_weight_threshold_sweep,table_2_weight_threshold_sweep.csv,table_2_weight_threshold_sweep.tex,12,4ac7ff29dffa38b1,a8491c69a0e3c594
3,table_3_sensor_gate_sweep,table_3_sensor_gate_sweep.csv,table_3_sensor_gate_sweep.tex,8,8fda9e883f01c420,8ca5de85231ee6fb
4,table_4_csi_gate_sweep,table_4_csi_gate_sweep.csv,table_4_csi_gate_sweep.tex,8,ca930b7d92736d2c,9a48c3b3182cc6b1
5,table_5_fusion_confusion,table_5_fusion_confusion.csv,table_5_fusion_confusion.tex,2,676396fc6f3dc5a8,4b0922e5565942c8
6,table_5_fusion_metrics,table_5_fusion_metrics.csv,table_5_fusion_metrics.tex,1,88571485e90c6c9c,32069916bbbdb254



Paper-facing files:
  table_1_fusion_by_class_state_wide.tex  (7 rows)
  table_1_fusion_by_class_state_long.tex  (14 rows)
  table_2_weight_threshold_sweep.tex  (12 rows)
  table_3_sensor_gate_sweep.tex  (8 rows)
  table_4_csi_gate_sweep.tex  (8 rows)
  table_5_fusion_confusion.tex  (2 rows)
  table_5_fusion_metrics.tex  (1 rows)


## 13c. Paper captions, files, and limitations

**Exported files** (under `CSI_TRAINING/models/fusion_v1/paper_tables/`):

| Paper table | Files | Rows |
|---|---|---|
| Class-conditional fusion rates | `table_1_fusion_by_class_state_wide.{csv,tex}` (paper), `..._long.{csv,tex}` | 7 / 14 |
| Weight × fused-threshold sweep | `table_2_weight_threshold_sweep.{csv,tex}` | 12 |
| Sensor-gate sweep | `table_3_sensor_gate_sweep.{csv,tex}` | 8 |
| CSI-gate sweep | `table_4_csi_gate_sweep.{csv,tex}` | 8 |
| Fusion confusion | `table_5_fusion_confusion.{csv,tex}` | 2 |
| Fusion metrics | `table_5_fusion_metrics.{csv,tex}` | 1 |

Metadata: `paper_tables_meta.json` (policy, fixture/model hashes, captions,
validation counts). The `.tex` files use `booktabs` rules
(`\toprule`/`\midrule`/`\bottomrule`); include `\usepackage{booktabs}` in the
paper preamble.

**Caption/limitation text.** The full captions are embedded in each `.tex`
file and in the metadata. They state that the CSI and sensor held-out sets are
from different sessions, that pairing is a class/state cross-product, that the
positive definition is CSI true class `smoke` AND sensor true state `smoke`,
and that only fusion decisions are counted. Key points for the paper text:

- 224 CSI windows (32/class, 7 classes) × 108 sensor intervals (73 no-smoke,
  35 smoke) = 24,192 simulated pairs; 1,120 positive (4.63% simulated
  prevalence), 23,072 negative.
- Deployed reference: 0.50 CSI / 0.50 sensor, fused threshold 0.70, sensor
  gate 0.404999, CSI argmax. Deployed results: TP 840, FP 175, FN 280,
  TN 22,897; TPR 0.750, FPR 0.008, J 0.742; precision 0.828, recall 0.750,
  F1 0.787, balanced accuracy 0.871.
- TPR/FPR/J and the per-class rates are the defensible metrics. Precision,
  F1, and balanced accuracy depend on the artificial 4.63% pair prevalence;
  they are in the CSV/metrics table for completeness, not as deployment
  estimates.
- The 24,192 pairs repeat 224 CSI windows and 108 sensor intervals
  (pseudo-replication). The class-conditional table reports the SD of each
  CSI window's fusion rate; do not compute naive pair-level confidence
  intervals or p-values.
- The deployed product also alerts on `smoke_only`; this section deliberately
  evaluates **fusion-only** alerts. The reported TPR is not the product's
  total alert sensitivity.
- The sensor-gate sweep is flat above ~0.45 because the 35 true-smoke
  intervals all clear the gate; at 0.405 the gate includes 37 intervals
  (35 true smoke + 2 no-smoke). The deployed gate is not re-tuned from these
  simulated pairs.
- No models were retrained; the fixture, model, calibrator, and policy hashes
  are recorded in `paper_tables_meta.json`.
